# Deep RecSys Course
## Домашнее задание 3: Нейросетевое ранжирование
### ФИО: Хамидуллин Амир 

В этом задании вы изучите способы кодирования вещественных и категориальных признаков, а затем обучите multi-task нейросетевую ранжирующую модель на основе DCN V2 со смесью экспертов.

Модель будет одновременно решать две задачи:
- предсказание полного прослушивания трека;
- предсказание лайка на трек.

Разбалловка

1. Подготовка датасета (1 балл)  
2. Обучение бейзлайнов и оценка качества (1 балл)  
3. Concat+MLP модель (3 балла)  
4. DCN-v2 (3 балла)  
5. Многоголовость (2 балла)  

# 1. Подготовка датасета (1 балл)

## yambda-50m-lag-features

Мы будем использовать датасет с уже заранее посчитанными признаками по Yambda.

Как был построен этот датасет:

1. За основу взят датасет Yambda `flat/50m multi_event`.
2. Из него были оставлены только неорганические события, то есть события, полученные с рекомендательных поверхностей, например из Моей Волны в Яндекс Музыке.
3. `is_skip = True`, если пользователь прослушал меньше половины трека.
4. `is_full_play = True`, если пользователь прослушал более 95% трека.
5. В Yambda лайк и прослушивание — это разные события. При этом модель обучается на событиях прослушивания: именно для них считаются признаки, и именно для них модель должна предсказывать таргеты. Поэтому лайки нужно атрибуцировать прослушиваниям, то есть понять, к какому именно прослушиванию относится каждый лайк. Для этого каждый лайк привязывается к ближайшему по времени прослушиванию того же `(uid, item_id)` в пределах 24 часов. Если подходящих прослушиваний два, выбирается ближайшее; при равенстве выбирается предыдущее. После этого `is_like = True` у прослушивания, если к нему был привязан хотя бы один лайк.
6. После этого по набору данных вычисляются вещественные счётчики. Они делятся на три типа: пользовательские, трековые и кросс-счётчики. Важно, что для каждого прослушивания с `timestamp` все счётчики считаются не в момент самого события, а в момент `timestamp - lag_seconds`, где `lag_seconds = 15 минут`. Такая задержка нужна, чтобы не допустить ликов при обучении и оценке. Идея в том, что при предсказании для текущего прослушивания модель не должна использовать информацию, которая появилась только после этого события или слишком близко к нему и в реальной системе ещё не успела бы попасть в признаки. Это особенно важно в нашей постановке, потому что далее качество будет оцениваться на соседних по времени прослушиваниях. Значит, при сравнении текущего трека с предыдущим нельзя допустить, чтобы признаки текущего объекта уже содержали фидбек на предыдущий трек, который к моменту предсказания ещё не должен был быть доступен. Иначе модель фактически будет подглядывать в ответ, а качество окажется завышенным. Кроме того, такой лаг лучше соответствует тому, как признаки обычно обновляются в реальных рекомендательных системах: не мгновенно, а с некоторой задержкой.
7. `is_like` и `is_full_play` используются как таргеты в задаче multi-task learning.

Код построения датасета можно посмотреть здесь: https://huggingface.co/datasets/matfu21/yambda-50m-lag-features/blob/main/event_processor.py#L296

In [1]:
import warnings
warnings.filterwarnings("ignore")

import gc
import polars as pl
import tests
from huggingface_hub import hf_hub_download

path = hf_hub_download(
    repo_id="matfu21/yambda-50m-lag-features",
    repo_type="dataset",
    filename="listens.parquet",
)
listens = pl.read_parquet(path)

## Оставляем только предпочтительные пары

В задачах ранжирования модель обычно обучается не на всех событиях подряд, а на специально отобранных объектах, которые несут сигнал о пользовательских предпочтениях. Это особенно важно для относительно небольших ранжирующих моделей. Если обучать такую модель на всех примерах подряд, она будет в большей степени учиться предсказывать глобальные вероятности лайка или полного прослушивания, чем правильно упорядочивать близкие по времени объекты. В результате качество именно на тех локальных попарных сравнениях, которые нас реально интересуют, может ухудшаться. Поэтому на практике датасеты для обучения ранжирования часто семплируются так, чтобы оставлять только примеры с полезным фидбеком.

При этом важно понимать, что это скорее инженерный компромисс, а не фундаментальное ограничение. Если бы у нас были очень большие модели с высокой выразительной способностью и без жёстких ограничений по вычислениям, то в идеале действительно хотелось бы учиться на всех доступных событиях. Такая модель могла бы одновременно хорошо восстанавливать глобальные закономерности и при этом качественно моделировать локальные предпочтения. Но в реальных рекомендательных системах ранжирующие модели обычно остаются относительно компактными, поэтому им полезно помогать фокусироваться именно на тех примерах, которые наиболее важны для ранжирования.

Почему ранжирующие модели обычно относительно небольшие: такие модели работают на финальном этапе ранжирования, где нужно очень быстро пересчитать score для большого числа кандидатов. У системы обычно есть жёсткие ограничения по latency, памяти и числу запросов в секунду. Поэтому слишком большие модели здесь использовать трудно: они будут слишком дорогими и медленными в продакшене. Именно из-за этого особенно важно, чтобы маленькая или средняя по размеру модель тратила свою выразительную способность на действительно полезные примеры, а не на предсказание усреднённых глобальных вероятностей.

В этом задании вам нужно оставить только те прослушивания, которые входят хотя бы в одну локальную пару с различающимся фидбеком. Идея такая: если у текущего трека и у соседнего по времени трека разный сигнал, то такая пара помогает обучать ранжирование. Если же и слева, и справа фидбек одинаковый, то текущее событие менее полезно для этой постановки.

### Что нужно сделать

Для каждого события нужно:
1. посмотреть на предыдущее событие того же пользователя;
2. посмотреть на следующее событие того же пользователя;
3. проверить, отличается ли `is_like` или `is_full_play` хотя бы с одной из сторон;
4. оставить только те строки, для которых такое отличие есть.

### Важная деталь про сортировку

Перед тем как смотреть на соседние события, данные нужно отсортировать по:
- `uid`,
- `timestamp`,
- `row_id`.

Поле `row_id` нужно для детерминированности. У одного и того же пользователя могут встретиться события с одинаковым `timestamp`, и тогда сортировка только по `uid` и `timestamp` не задаёт однозначный порядок. Добавление `row_id` делает этот порядок фиксированным и воспроизводимым.

In [2]:
import polars as pl

# # В одном пайплайне: создаем index -> сортируем -> создаем соседей -> выводим
# demo = (
#     listens
#     .with_row_index("row_id")
#     .sort(["uid", "timestamp", "row_id"])
#     .with_columns(
#         pl.col("is_like").shift(1).over("uid").alias("prev_is_like"),
#         pl.col("is_like").shift(-1).over("uid").alias("next_is_like")
#     )
#     .select([
#         "row_id", "uid", "timestamp", "is_like", "prev_is_like", "next_is_like"
#     ])
# )

# demo.head(20)


In [3]:
import polars as pl
def extract_preference_pairs(df: pl.DataFrame) -> pl.DataFrame:
    return(
        df.with_row_index("row_id").sort(["uid", "timestamp", "row_id"]).with_columns(pl.col("is_like").shift(1).over("uid").alias("prev_is_like"),
                                                                        (pl.col("is_like").shift(-1).over("uid").alias("next_is_like")),
                                                                        (pl.col("is_full_play").shift(1).over("uid").alias("prev_is_full_play")),
                                                                        (pl.col("is_full_play").shift(-1).over("uid").alias("next_is_full_play"))
    ).filter(
        (pl.col("is_like") != pl.col("prev_is_like")) | 
        (pl.col("is_like") != pl.col("next_is_like")) |
        (pl.col("is_full_play") != pl.col("prev_is_full_play")) |
        (pl.col("is_full_play") != pl.col("next_is_full_play"))
    ).drop(["prev_is_like", "next_is_like", 
            "prev_is_full_play", "next_is_full_play",
            "row_id"])
    )

listens = extract_preference_pairs(listens)
tests.test_extract_preference_pairs(listens)

All good! :)


## Набор данных Yambda

В этом задании мы будем использовать датасет Yambda: https://huggingface.co/datasets/yandex/yambda. Из него мы получим информацию об альбомах, к которым относится трек, а также об артистах этого трека. Эти данные понадобятся вам для построения мультивалентных признаков.

Ниже приведён класс из Hugging Face, который вам предстоит использовать.

In [4]:
from dataclasses import dataclass


@dataclass
class DatasetConfig:
    dataset_type: str = 'flat'
    dataset_size: str = '50m'
    interaction_name: str = 'multi_event'
    default_like_window_seconds: int = 24 * 60 * 60
    lag_seconds: int = 15 * 60

dataset_config = DatasetConfig()

In [5]:
from typing import Literal
from datasets import Dataset, DatasetDict, load_dataset


# YambdaDataset wrapper class (https://huggingface.co/datasets/yandex/yambda)
class YambdaDataset:
    INTERACTIONS = frozenset([
        "likes", "listens", "multi_event", "dislikes", "unlikes", "undislikes"
    ])

    def __init__(
        self,
        dataset_type: Literal["flat", "sequential"] = "flat",
        dataset_size: Literal["50m", "500m", "5b"] = "50m"
    ):
        assert dataset_type in {"flat", "sequential"}
        assert dataset_size in {"50m", "500m", "5b"}
        self.dataset_type = dataset_type
        self.dataset_size = dataset_size

    def interaction(self, event_type: Literal[
        "likes", "listens", "multi_event", "dislikes", "unlikes", "undislikes"
    ]) -> Dataset:
        assert event_type in YambdaDataset.INTERACTIONS
        return self._download(f"{self.dataset_type}/{self.dataset_size}", event_type)

    def audio_embeddings(self) -> Dataset:
        return self._download("", "embeddings")

    def album_item_mapping(self) -> Dataset:
        return self._download("", "album_item_mapping")

    def artist_item_mapping(self) -> Dataset:
        return self._download("", "artist_item_mapping")

    @staticmethod
    def _download(data_dir: str, file: str) -> Dataset:
        data = load_dataset("yandex/yambda", data_dir=data_dir, data_files=f"{file}.parquet")
        # Returns DatasetDict; extracting the only split
        assert isinstance(data, DatasetDict)
        return data["train"]


In [6]:
yambda_dataset = YambdaDataset(
    dataset_type=dataset_config.dataset_type,
    dataset_size=dataset_config.dataset_size
)
albums = yambda_dataset.album_item_mapping().to_polars()
artists = yambda_dataset.artist_item_mapping().to_polars()

In [7]:
print(albums.head(5))
print(artists.head(5))
print(listens.head(5))

shape: (5, 2)
┌──────────┬─────────┐
│ album_id ┆ item_id │
│ ---      ┆ ---     │
│ u32      ┆ u32     │
╞══════════╪═════════╡
│ 1        ┆ 1491131 │
│ 1        ┆ 5109849 │
│ 1        ┆ 6735246 │
│ 2        ┆ 2859065 │
│ 3        ┆ 1859377 │
└──────────┴─────────┘
shape: (5, 2)
┌───────────┬─────────┐
│ artist_id ┆ item_id │
│ ---       ┆ ---     │
│ u32       ┆ u32     │
╞═══════════╪═════════╡
│ 1         ┆ 953587  │
│ 1         ┆ 1921481 │
│ 1         ┆ 2659068 │
│ 1         ┆ 2740764 │
│ 1         ┆ 6767564 │
└───────────┴─────────┘
shape: (5, 23)
┌─────┬───────────┬─────────┬─────────────┬───┬─────────────┬────────────┬────────────┬────────────┐
│ uid ┆ timestamp ┆ item_id ┆ played_rati ┆ … ┆ ui_lag_skip ┆ user_lag_a ┆ item_lag_a ┆ ui_lag_avg │
│ --- ┆ ---       ┆ ---     ┆ o_pct       ┆   ┆ _cnt        ┆ vg_played_ ┆ vg_played_ ┆ _played_ra │
│ u32 ┆ u32       ┆ u32     ┆ ---         ┆   ┆ ---         ┆ ratio      ┆ ratio      ┆ tio        │
│     ┆           ┆         ┆ u16   

## Добавляем мультивалентные признаки

Ранее мы загрузили данные `albums` и `artists`. Теперь на их основе нужно добавить в таблицу две новые колонки с мультивалентными категориальными признаками:
- `artist_ids` — список артистов, соответствующих данному треку;
- `album_ids` — список альбомов, соответствующих данному треку.

Таким образом, для каждого трека в этих колонках должен храниться не один идентификатор, а список идентификаторов.

In [7]:
def join_item_artist_album(
    listens: pl.DataFrame,
    artists: pl.DataFrame,
    albums: pl.DataFrame,
) -> pl.DataFrame:

    artists_agg = artists.group_by("item_id").agg(pl.col("artist_id").alias("artist_ids"))
    albums_agg = albums.group_by("item_id").agg(pl.col("album_id").alias("album_ids"))
    df = listens.join(artists_agg, on="item_id", how = "left").join(albums_agg, on ="item_id", how="left")
    return df

listens = join_item_artist_album(listens, artists, albums)
tests.test_join_item_artist_album(listens)
del artists, albums
gc.collect()

All good! :)


0

## Разбиваем данные на обучающую и тестовую выборки

Далее вам нужно написать функцию, которая делит датасет на две части по времени:
- в `train` попадают все события, кроме последних `test_last_seconds` секунд;
- в `test` попадают события из последних `test_last_seconds` секунд, включая границу.

В этом задании в качестве тестовой выборки мы будем использовать последние 30 дней.

In [8]:
def temporal_train_test_split(
    df: pl.DataFrame,
    test_last_seconds: float,
    time_column: str = "timestamp",
) -> tuple[pl.DataFrame, pl.DataFrame]:

    max_time = df.select(pl.col(time_column).max()).item()
    split_time = max_time - test_last_seconds

    train_df = df.filter(pl.col(time_column) < split_time)
    test_df = df.filter(pl.col(time_column) >= split_time)
    
    
    return [train_df, test_df]

train_listens, test_listens = temporal_train_test_split(listens, test_last_seconds=30 * 24 * 60 * 60)
tests.test_temporal_train_test_split(train_listens, test_listens)
del listens
gc.collect()

All good! :)


0

In [9]:
print(f"train: {train_listens.shape}")
print(f"test: {test_listens.shape}")


train: (5955066, 25)
test: (930406, 25)


# 2. Обучение бейзлайнов и оценка качества (1 балл)

## Pairwise accuracy

Метрика, которой будет оцениваться качество вашего решения, называется `pairwise accuracy`.

Идея метрики такая: мы хотим проверить, умеет ли модель правильно упорядочивать соседние по времени события одного пользователя. Например, если в паре соседних треков на один был лайк, а на другой нет, то трек с лайком должен получить больший скор.

Функция получает на вход:
- `uids` — идентификаторы пользователей;
- `timestamps` — времена событий;
- `labels` — бинарные таргеты (`0` или `1`);
- `probs` — предсказанные моделью скоры;
- `session_gap_seconds` — максимальный допустимый разрыв по времени между соседними событиями.

Что нужно реализовать:
1. сгруппировать события по пользователям;
2. внутри каждого пользователя отсортировать события по времени;
3. рассмотреть только соседние пары;
4. отбросить пары, у которых разница по времени больше `session_gap_seconds`;
5. оставить только пары, у которых различаются `label`;
6. проверить, совпадает ли порядок по `probs` с порядком по `label`.

Вклад одной пары:
- 1, если модель ранжирует пару правильно;
- 0, если неправильно;
- 0.5, если скоры равны.

Формально, для соседней валидной пары $(i, i+1)$:

$$
m(i, i+1)=
\begin{cases}
1, & \text{если } (y_{i+1}-y_i)(s_{i+1}-s_i) > 0, \\
0.5, & \text{если } s_{i+1} = s_i, \\
0, & \text{иначе.}
\end{cases}
$$

Итоговая метрика:

$$
\text{PairwiseAccuracy} =
\frac{1}{|P|}\sum_{(i,i+1)\in P} m(i,i+1),
$$

где $P$ — множество всех валидных соседних пар по всем пользователям.

Почему мы смотрим именно на соседние пары: в рекомендательных задачах контекст пользователя быстро меняется. Если сравнивать два события, между которыми прошло много времени, такое сравнение уже может быть некорректным: пользователь был в другом состоянии, слушал другую музыку и вообще решал другую задачу. Поэтому мы сравниваем только локальные, соседние по времени события.

По этой же причине здесь не очень подходит ROC AUC. Эта метрика фактически сравнивает позитивные и негативные объекты между собой глобально, в том числе из совершенно разных моментов времени. В результате модель может получить хороший ROC AUC, даже если она плохо упорядочивает близкие по времени события внутри реальной пользовательской последовательности. `Pairwise accuracy` лучше отражает именно качество локального ранжирования, которое важно в этой задаче.

Дополнительно важно, что временной порог в метрике согласован с лагом, который использовался при построении признаков. В признаках мы используем задержку 15 минут, чтобы модель не могла опираться на слишком свежий фидбек, который в реальной системе ещё не успел бы попасть в фичи. В метрике используется тот же порог: если два соседних события находятся друг от друга дальше, чем на 15 минут, мы их не сравниваем. Это делает постановку более согласованной: и при построении признаков, и при оценке качества мы работаем в одном и том же локальном временном окне.

Функция получает на вход:
- `uids` — идентификаторы пользователей;
- `timestamps` — времена событий;
- `labels` — бинарные таргеты (`0` или `1`);
- `probs` — предсказанные моделью скоры;
- `session_gap_seconds` — максимальный допустимый разрыв по времени между соседними событиями.

Что нужно реализовать:
1. сгруппировать события по пользователям;
2. внутри каждого пользователя отсортировать события по времени;
3. рассмотреть только соседние пары;
4. отбросить пары, у которых разница по времени больше `session_gap_seconds`;
5. оставить только пары, у которых различаются `label`;
6. проверить, совпадает ли порядок по `probs` с порядком по `label`.

In [10]:
import numpy as np


def compute_pairwise_accuracy(
    uids: np.array,
    timestamps: np.array,
    labels: np.array,
    probs: np.array, # scores
    session_gap_seconds: int = 15 * 60,
) -> dict[str, float]:
    
    events_by_users = np.lexsort((timestamps, uids)) # список из индексов
    uids_sort = uids[events_by_users]
    timestamps_sort = timestamps[events_by_users]
    labels_sort = labels[events_by_users]
    probs_sort = probs[events_by_users]

    uid_curr, ts_curr, y_curr, s_curr = uids_sort[:-1], timestamps_sort[:-1], labels_sort[:-1], probs_sort[:-1]
    uid_next, ts_next, y_next, s_next = uids_sort[1:],  timestamps_sort[1:],  labels_sort[1:],  probs_sort[1:]

    mask = ((uid_curr == uid_next) & (ts_next - ts_curr <= session_gap_seconds) & (y_curr != y_next))

    y_curr, y_next = y_curr[mask], y_next[mask]
    s_curr, s_next = s_curr[mask], s_next[mask]

    val = (y_next.astype(int) - y_curr.astype(int)) * (s_next - s_curr)
    # val = val[val>0]

    m = np.zeros_like(y_curr, dtype=float)
    m[val > 0] = 1
    m[s_next - s_curr == 0] = 0.5
    
    return m.mean() #np.mean(m)

tests.test_pairwise_accuracy(compute_pairwise_accuracy)

All good! :)


In [11]:
# import numpy as np
# uids       = [2,    1,    2 , 3  ] # Юзеры
# timestamps = [5000, 100000000, 2000, 500] # Время

# x = np.lexsort((timestamps,uids ))
# print(np.lexsort((timestamps,uids )))

# a = np.array([1,2,3,4])
# print(a[x])

# b = np.array([5,6,7])
# c = np.array([6,7, 8])
# print((b == c))

# mask = [True, False,False]
# c = np.array([1,2,3])
# print(c[mask])


# a = np.array([1,2,3])
# c = np.array([1,20,3])
# print(a * c)

# a = a[a > 1]
# print(a)

scores = np.random.random(5)
scores

array([0.04242835, 0.88682122, 0.99801919, 0.96584774, 0.49861093])

## Считаем метрики случайного бейзлайна

Перед обучением модели полезно посчитать качество простого случайного бейзлайна. Для этого сгенерируйте для каждого объекта в тестовой выборке случайный скор и посчитайте по нему обе метрики `pairwise accuracy`:
- по таргету `is_like`;
- по таргету `is_full_play`.

На этом шаге нужно:
1. взять тестовую выборку;
2. сгенерировать для каждого объекта случайное значение score;
3. вычислить `pair_accuracy_like`;
4. вычислить `pair_accuracy_full_play`.

В результате у вас должен получиться словарь `metrics` с двумя значениями метрик для случайного ранжирования.

In [12]:
uids = test_listens["uid"].to_numpy()
ts = test_listens["timestamp"].to_numpy()
labels_like = test_listens["is_like"].to_numpy()
labels_full_play = test_listens["is_full_play"].to_numpy()
np.random.seed(42)
scores = np.random.random(len(uids))


metrics = {
    "pair_accuracy_like": compute_pairwise_accuracy(uids, ts, labels_like, scores),
    "pair_accuracy_full_play": compute_pairwise_accuracy(uids, ts, labels_full_play, scores)
}
print(metrics)
tests.test_random_baseline_metrics(metrics)

{'pair_accuracy_like': np.float64(0.5033185983271687), 'pair_accuracy_full_play': np.float64(0.4996053599497365)}
All good! :)


## Считаем метрики popularity baseline

Ещё один простой бейзлайн — ранжировать треки по их популярности в обучающей выборке. Для этого посчитайте, сколько раз каждый `item_id` встретился в `train`, и используйте это значение как score на `test`.

На этом шаге нужно:
1. по обучающей выборке посчитать популярность каждого трека (сколько раз он встретился);
2. присоединить эту популярность к объектам тестовой выборки;
3. для треков, которых не было в `train`, подставить `0`;
4. вычислить `pair_accuracy_like`;
5. вычислить `pair_accuracy_full_play`.

В результате у вас должен получиться словарь `metrics` с двумя значениями метрик для popularity baseline.

In [13]:
item_popularity = train_listens.group_by("item_id").agg(pl.len().alias("popularity"))

test_with_pop = test_listens.join(item_popularity, on="item_id", how="left").with_columns(pl.col("popularity").fill_null(0))
uids = test_with_pop["uid"].to_numpy()
ts = test_with_pop["timestamp"].to_numpy()

labels_like = test_with_pop["is_like"].to_numpy()
labels_full_play = test_with_pop["is_full_play"].to_numpy()
scores = test_with_pop["popularity"].to_numpy().astype(float)


metrics = {
    "pair_accuracy_like": compute_pairwise_accuracy(uids, ts, labels_like, scores),
    "pair_accuracy_full_play": compute_pairwise_accuracy(uids, ts, labels_full_play, scores)
}
print(metrics)
tests.test_popularity_baseline_metrics(metrics)

{'pair_accuracy_like': np.float64(0.48755686724317915), 'pair_accuracy_full_play': np.float64(0.5246778596870157)}
All good! :)


## Обучаем CatBoost baseline

Теперь обучим ещё один бейзлайн — модель CatBoost. В отличие от предыдущих простых бейзлайнов, здесь модель уже будет использовать признаки из датасета и учиться предсказывать таргет по обучающей выборке.

На этом шаге нужно:
1. взять вещественные признаки `DENSE_COLUMNS` и категориальные признаки `SPARSE_COLUMNS`;
2. обучить `CatBoostClassifier` на задаче предсказания `is_full_play`;
3. получить предсказанные вероятности на тестовой выборке;
4. использовать вероятность полного прослушивания как score;
5. посчитать `pair_accuracy_like`;
6. посчитать `pair_accuracy_full_play`.

Обратите внимание, что модель обучается только на таргете `is_full_play`, но качество нужно измерить по обоим таргетам. Это позволит понять, насколько предсказание лайка переносится на задачу ранжирования по полному прослушиванию.

В результате у вас должен получиться словарь `metrics` с двумя значениями:
- `pair_accuracy_like`;
- `pair_accuracy_full_play`.

In [14]:
%pip install catboost

Note: you may need to restart the kernel to use updated packages.


In [15]:
%pip install scikit-learn


Note: you may need to restart the kernel to use updated packages.


In [16]:
train_listens.head()

uid,timestamp,item_id,played_ratio_pct,track_length_seconds,is_like,is_full_play,is_skip,user_lag_listen_cnt,user_lag_like_cnt,user_lag_full_play_cnt,user_lag_skip_cnt,item_lag_listen_cnt,item_lag_like_cnt,item_lag_full_play_cnt,item_lag_skip_cnt,ui_lag_listen_cnt,ui_lag_like_cnt,ui_lag_full_play_cnt,ui_lag_skip_cnt,user_lag_avg_played_ratio,item_lag_avg_played_ratio,ui_lag_avg_played_ratio,artist_ids,album_ids
u32,u32,u32,u16,u32,bool,bool,bool,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,list[u32],list[u32]
100,40110,732449,100,240,false,true,false,0.0,0.0,0.0,0.0,2.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,50.5,0.0,[94924],[1632076]
100,40360,3397170,46,130,false,false,true,2.0,0.0,2.0,0.0,4.0,0.0,3.0,1.0,0.0,0.0,0.0,0.0,100.0,83.25,0.0,[997338],[403561]
100,40380,7849270,100,205,false,true,false,2.0,0.0,2.0,0.0,7.0,0.0,4.0,3.0,0.0,0.0,0.0,0.0,100.0,57.428571,0.0,[995468],[477338]
100,41130,6474571,100,245,false,true,false,4.0,0.0,4.0,0.0,2.0,1.0,0.0,2.0,0.0,0.0,0.0,0.0,100.0,7.0,0.0,[163539],[1412710]
100,41545,7847600,27,250,false,false,true,7.0,0.0,6.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,92.285714,100.0,0.0,[639419],[230968]


In [17]:
from sklearn.model_selection import train_test_split
from catboost import CatBoostClassifier

DENSE_COLUMNS: tuple[str, ...] = (
    "user_lag_listen_cnt",
    "user_lag_like_cnt",
    "user_lag_full_play_cnt",
    "user_lag_skip_cnt",
    "item_lag_listen_cnt",
    "item_lag_like_cnt",
    "item_lag_full_play_cnt",
    "item_lag_skip_cnt",
    "ui_lag_listen_cnt",
    "ui_lag_like_cnt",
    "ui_lag_full_play_cnt",
    "ui_lag_skip_cnt",
    "user_lag_avg_played_ratio",
    "item_lag_avg_played_ratio",
    "ui_lag_avg_played_ratio",
)
MULTIVALENT_COLUMNS: tuple[str, ...] = ("artist_ids", "album_ids")
SPARSE_COLUMNS = ("uid", "item_id")
LABEL_COLUMNS = ("is_like", "is_full_play")

FEATURES = list(DENSE_COLUMNS) + list(SPARSE_COLUMNS)

X_train = train_listens.select(FEATURES).to_pandas()
y_train = train_listens["is_full_play"].to_pandas()
X_test = test_listens.select(FEATURES).to_pandas()

best_model = CatBoostClassifier(
   iterations=500,
   l2_leaf_reg=4.0, 
   cat_features=list(SPARSE_COLUMNS), 
   task_type="CPU",
   verbose=100  #  печатать лог каждые 100 деревьев
)

best_model.fit(X_train, y_train,plot=True)


uids = test_listens["uid"].to_numpy()
ts = test_listens["timestamp"].to_numpy()
labels_like = test_listens["is_like"].to_numpy()
labels_full_play = test_listens["is_full_play"].to_numpy()


scores = best_model.predict_proba(X_test)[:, 1]


metrics = {
    "pair_accuracy_like": compute_pairwise_accuracy(
        uids, ts, labels_like, scores
    ),
    
    "pair_accuracy_full_play": compute_pairwise_accuracy(
        uids, ts, labels_full_play, scores
    )
}

tests.test_catboost_baseline_metrics(metrics)

MetricVisualizer(layout=Layout(align_self='stretch', height='500px'))

0:	learn: 0.6919372	total: 1.3s	remaining: 10m 50s


KeyboardInterrupt: 

Он работает просто случайно запустил заново а времени переучивать нет

In [ ]:
print(metrics)

{'pair_accuracy_like': np.float64(0.48726689263206735), 'pair_accuracy_full_play': np.float64(0.5910422298392597)}


# 3. Concat+MLP модель (3 балла)

В этом задании вам предстоит реализовать первое нейросетевое решение.

Идея модели очень простая:
1. вещественные признаки кодируются с помощью `PiecewiseLinearEncoder`;
2. категориальные и мультивалентные признаки кодируются с помощью `Multisize Unified Embeddings`;
3. все полученные представления конкатенируются в один вектор;
4. этот вектор подаётся в простую полносвязную сеть с `ReLU` в качестве нелинейности.

На этом этапе модель предсказывает только вероятность полного прослушивания, то есть решается одна бинарная задача. Пока что речь о multi-task learning и нескольких головах не идёт.

Таким образом, вам нужно реализовать модель, которая:
- получает dense-, sparse- и multivalent-признаки;
- кодирует каждый тип признаков своим энкодером;
- объединяет все представления;
- пропускает результат через `MLP`;
- выдаёт скалярный score для предсказания полного прослушивания.

## Готовим датасет для обучения

Теперь нужно реализовать класс `RankerDataset`, который будет подготавливать батчи для обучения нейросетевой модели.

В этом задании батчевание происходит **на этапе самого датасета**, а не в `DataLoader`. Здесь это сделано специально: табличные данные уже лежат целиком в памяти, а модель сама по себе небольшая, поэтому узким местом легко становится именно подготовка батчей и перекладывание данных. Если формировать большие батчи на лету, то даже при больших `num_workers` и `prefetch_factor` загрузка данных может стать bottleneck. Поэтому здесь удобнее заранее нарезать датафрейм на батчи и хранить их в готовом виде внутри датасета.

Вам дан каркас класса `RankerDataset`. Нужно реализовать его так, чтобы в конструкторе датафрейм разбивался на батчи размера `batch_size`, а каждый батч преобразовывался в словарь фиксированной структуры.

### Что нужно сделать

1. Проверить, что `batch_size >= 1`.
2. Разбить входной `df` на последовательные батчи размера `batch_size`.
3. Для каждого батча собрать dense-, sparse- и multivalent-признаки, а также таргеты и мета-информацию.
4. Последовательно применить к каждому батчу все функции из `transforms`.
5. Сохранить готовые батчи внутри датасета.
6. Реализовать:
   - `__len__` — количество батчей;
   - `__getitem__` — получение батча по индексу.

### Формат батча

Каждый батч должен быть словарём со следующими ключами:
- `labels`
- `dense_features`
- `sparse_features`
- `multivalent_features`
- `meta`

#### `labels`

Словарь, где ключами являются названия таргетов из `label_columns`, а значениями — одномерные `torch.Tensor` длины `batch_size`.

Пример:

```python
{
    "is_like": ...,
    "is_full_play": ...,
}
```

#### `dense_features`
Тензор размера [batch_size, num_dense_features] типа torch.float32.

#### `sparse_features`
Словарь, где для каждого имени из sparse_columns хранится одномерный тензор длины batch_size.

Пример:

```python
{
    "uid": ...,
    "item_id": ...,
}
```

#### `multivalent_features`
Словарь, где для каждого мультивалентного признака нужно сохранить ещё один словарь с двумя полями:
-  values — один общий плоский тензор со всеми значениями из списков;
- lengths — тензор длин списков для каждого объекта в батче.

Пример:

```python
{
    "artist_ids": {
        "values": ...,
        "lengths": ...,
    },
    "album_ids": {
        "values": ...,
        "lengths": ...,
    },
}
```

Именно в таком формате мультивалентные признаки удобно потом агрегировать внутри модели.

#### `meta`
Словарь с технической информацией о батче:

```python
{
    "timestamp": ...,
    "uid": ...,
    "item_id": ...,
}
```

Что такое transforms

Аргумент transforms — это список функций, которые принимают батч и возвращают преобразованный батч. Эти преобразования нужно применять последовательно к каждому уже собранному батчу.

То есть логика такая:
1. вы собрали батч в нужном формате;
2.	прогнали его через все функции из transforms;
3.	сохранили итоговый батч в датасет.

Это позволит дальше независимо добавлять, например, перенос на устройство, нормализацию, переименование полей или любые другие постобработки.

Дополнительные замечания
- Dense-признаки и таргеты нужно привести к torch.float32.
- Для мультивалентных признаков lengths и values должны быть целочисленными тензорами.
- Последний батч может быть неполным.
- Пустые батчи сохранять не нужно.

В результате ваш RankerDataset должен возвращать готовые батчи, а не отдельные объекты.

In [ ]:
# import torch
# df = pl.DataFrame(
#     {
#         "foo": [1, 2, 3,4,5,6,7,8],
#         "ç": [1, 2, 3,4,5,6,7,8],
#         "ham": [1, 2, 3,4,5,6,7,8],
#         "barfefe": [1, 2, 3,4,5,6,7,8],
#         "barecfew": [1, 2, 3,4,5,6,7,8],
#         "barecfefefew": [1, 2, 3,4,5,6,7,8],
#     }
# )
# dense_columns = ["ham", "barecfew"]

# for start_indx in range(0, len(df), 2):
    
#     chunk = df.slice(start_indx, 2)
    
#     dense_features = torch.tensor([chunk[col] for col in dense_columns],dtype = torch.float32)
#     print(dense_features)
#     break


a = [[1,2,3],[3],[4,5,6,7]]
a = pl.Series(a)
a = a.list.len().to_numpy()
print(a)

[3 1 4]


In [18]:
import polars as pl
from typing import Any, Callable
import torch
from torch.utils.data import Dataset


class RankerDataset(Dataset):
    def __init__(
        self,
        df: pl.DataFrame,
        transforms: list[Callable[[Any], Any]],
        label_columns: list[str],
        dense_columns: list[str],
        sparse_columns: list[str],
        multivalent_columns: list[str],
        batch_size: int,
    ):

        assert batch_size >= 1, "Размер батча должен быть >= 1"
        self.batches = []
        self.batch_size = batch_size 
        meta_columns = ["timestamp", "uid", "item_id"]


        for start_indx in range(0,len(df), self.batch_size):
            chunk = df.slice(start_indx, self.batch_size)

            labels = {col: torch.tensor(chunk[col].to_numpy(), dtype = torch.float32) for col in label_columns}
            dense_features = torch.tensor([chunk[col].to_numpy() for col in dense_columns], dtype = torch.float32).T
            # sparse_features = {col: torch.tensor(chunk[col].to_numpy()) for col in sparse_columns}
            # multivalent_features = {col: {
            #     "values": torch.tensor(chunk[col].explode().to_numpy()),
            #     "lengths": torch.tensor(chunk[col].list.len().to_numpy())
            # } 
            # for col in multivalent_columns}
            # meta = {col: torch.tensor(chunk[col].to_numpy()) for col in meta_columns}
            # Для sparse:
            sparse_features = {
                col: torch.tensor(chunk[col].to_numpy().astype(np.int64), dtype=torch.long) 
                for col in sparse_columns
            }
            multivalent_features = {col: {
                "values": torch.tensor(chunk[col].explode().to_numpy().astype(np.int64), dtype=torch.long),
                "lengths": torch.tensor(chunk[col].list.len().to_numpy().astype(np.int64), dtype=torch.long)
            } for col in multivalent_columns}

            meta = {
                col: torch.tensor(chunk[col].to_numpy().astype(np.int64), dtype=torch.long) 
                for col in meta_columns
            }


            batch_dict = {
                "labels": labels,
                "dense_features": dense_features,
                "sparse_features" : sparse_features,
                "multivalent_features" : multivalent_features,
                "meta": meta

            }

            for transform in transforms: 
                batch_dict = transform(batch_dict)
            
            self.batches.append(batch_dict)
            

    def __len__(self) -> int:
        return len(self.batches)

    def __getitem__(self, idx: int) -> dict[str, Any]:
        return self.batches[idx]

tests.test_ranker_dataset(RankerDataset)

All good! :)


## Multihash transform

Теперь реализуем `MultihashTransform` — преобразование, которое будет переводить sparse- и multivalent-признаки в набор хешей. Этот шаг нужен для дальнейшей работы с unified embeddings.

Идея здесь следующая: вместо того, чтобы хранить отдельную embedding-таблицу под каждый признак или использовать один-единственный хеш, мы отображаем каждый идентификатор сразу в несколько хешей с разными seed. В результате один и тот же исходный ID превращается в небольшой набор индексов. Такой подход предлагается в статье [Unified Embedding: Battle-Tested Feature Representations for Web-Scale ML Systems](https://arxiv.org/pdf/2305.12102) и позволяет компактнее работать с большим количеством категориальных признаков.

Вам дан каркас класса `MultihashTransform`. Нужно реализовать его так, чтобы он применял multihash-преобразование:
- к sparse-признакам;
- к multivalent-признакам.

### Что приходит на вход

На вход `__call__` подаётся `sample` — словарь батча, в котором уже есть поля:
- `sparse_features`;
- `multivalent_features`.

Именно их и нужно преобразовать.

### Что задаётся в конструкторе

- `sparse_features_config` — словарь вида  
  `{feature_name: [seed1, seed2, ...]}`  
  для sparse-признаков;
- `sparse_features_name` — имя поля в `sample`, где лежат sparse-признаки;
- `multivalent_features_config` — словарь такого же вида для multivalent-признаков;
- `multivalent_features_name` — имя поля в `sample`, где лежат multivalent-признаки;
- `cardinality` — размер общего хеш-пространства, то есть все индексы после хеширования должны лежать в диапазоне `[0, cardinality)`.

### Что нужно сделать

Для каждого sparse-признака:
1. взять тензор идентификаторов;
2. для каждого `seed` из конфига отдельно посчитать хеш;
3. привести хеш в диапазон `[0, cardinality)` с помощью взятия остатка;
4. сохранить результат обратно в `sample`.

Для каждого multivalent-признака:
1. взять плоский тензор `values`;
2. для каждого `seed` из конфига отдельно посчитать хеш;
3. привести хеш в диапазон `[0, cardinality)`;
4. сохранить результат обратно в `sample`, не меняя `lengths`.

### Какой формат должен получиться

Если исходный sparse-признак имел форму

```python
[batch_size]
```

то после multihash-преобразования он должен иметь форму

```python
[batch_size, num_hashes]
```

где num_hashes — число seed для этого признака.

Если values у multivalent-признака имели форму

```python
[num_values]
```

то после преобразования должно получиться

```python
[num_values, num_hashes]
```

Поле lengths у multivalent-признаков менять не нужно.

Важные замечания
- Для хеширования нужно использовать mmh3.
- Один и тот же исходный ID при одном и том же seed всегда должен давать один и тот же индекс.
- Разные seed должны давать разные хеш-проекции одного и того же ID.
- Все полученные индексы должны быть целыми числами из диапазона `[0, cardinality)`.
- Преобразование должно происходить in-place: нужно модифицировать sample и вернуть его обратно.

Итоговая идея такая: после MultihashTransform каждый категориальный ID заменяется не одним индексом, а набором индексов, полученных несколькими независимыми хеш-функциями.


In [19]:
%pip install mmh3

Note: you may need to restart the kernel to use updated packages.


In [20]:
from typing import Any
import mmh3


class MultihashTransform:
    def __init__(
        self,
        sparse_features_config: dict,
        sparse_features_name: str,
        multivalent_features_config: dict,
        multivalent_features_name: str,
        cardinality: int,
    ):
        self.sparse_features_config = sparse_features_config
        self.sparse_features_name = sparse_features_name
        self.multivalent_features_config = multivalent_features_config
        self.multivalent_features_name = multivalent_features_name
        self.cardinality = cardinality


    def __call__(self, sample: dict[str, Any]) -> dict[str, Any]:
        sparse_dict = sample[self.sparse_features_name] # батч с нужными колонками

        for feature_name, seeds in self.sparse_features_config.items():
            info = sparse_dict[feature_name]
            
            res_hes = []
            for el in info:
                hash_per_el = []
                for seed in seeds:
                    hash = mmh3.hash(str(int(el)), int(seed)) % self.cardinality # число - адресс 
                    hash_per_el.append(hash)
                res_hes.append(hash_per_el)
            
            res_hes = torch.tensor(res_hes, dtype = torch.long)
            sparse_dict[feature_name] = res_hes # [[num1,num2],[],....] 
        
        
        multivalent_dict = sample[self.multivalent_features_name]
        for multivalnet_col, seeds in self.multivalent_features_config.items():
            values_tensor = multivalent_dict[multivalnet_col]["values"]
            res_hes = []
            for el in values_tensor:
                hash_per_el = []
                for seed in seeds:
                    hash = mmh3.hash(str(int(el)),int(seed)) % self.cardinality
                    hash_per_el.append(hash)
                res_hes.append(hash_per_el)
        
            res_hes = torch.tensor(res_hes, dtype = torch.long)
            multivalent_dict[multivalnet_col]["values"] = res_hes
                
        
        return sample


tests.test_multihash_transform(MultihashTransform)

All good! :)


## CategoricalEncoder

Теперь реализуем `CategoricalEncoder` — простой энкодер категориальных признаков. Его задача состоит в том, чтобы по входным индексам возвращать соответствующие embedding-векторы.

В конструктор класса передаётся объект `nn.Embedding`, который нужно сохранить внутри модуля. В методе `forward` на вход подаётся тензор `ids`, а на выходе должен получаться результат применения `embeddings(ids)`.

Иными словами, `CategoricalEncoder` — это тонкая обёртка над `nn.Embedding`.

### Что нужно реализовать

1. Сохранить переданный слой `embeddings`.
2. В `forward` применить его к тензору `ids`.
3. Вернуть получившийся тензор.

### Ожидаемое поведение

Если на вход подан тензор индексов формы

```python
[batch_size]
```

то на выходе должен получиться тензор формы

```python
[batch_size, embedding_dim]
```

Если на вход подан тензор формы

```python
[batch_size, num_hashes]
```

то на выходе должен получиться тензор формы

```python
[batch_size, num_hashes, embedding_dim]
```

То есть энкодер должен работать с произвольной входной формой так же, как обычный nn.Embedding.


In [21]:
a = torch.tensor([[1,2,3],[4,5,6]])
print(a.ndim)

2


In [22]:
import torch
import torch.nn as nn


class CategoricalEncoder(nn.Module):
    def __init__(self, embeddings: nn.Embedding):
        super().__init__()
        self.embeddings = embeddings


    def forward(self, ids: torch.Tensor) -> torch.Tensor:
        return self.embeddings(ids)


tests.test_categorical_encoder(CategoricalEncoder)

All good! :)


## MultivalentEncoder

Теперь реализуем `MultivalentEncoder` — энкодер для мультивалентных категориальных признаков. В отличие от обычного категориального признака, здесь каждому объекту соответствует не один ID, а список ID, например список артистов или альбомов.

В конструктор класса передаётся объект `nn.Embedding`, который нужно сохранить внутри модуля. В методе `forward` на вход подаются:
- `ids` — плоский тензор идентификаторов после `MultihashTransform`;
- `lengths` — длины списков для каждого объекта в батче.

Нужно получить embedding для каждого ID, а затем усреднить эмбеддинги внутри каждого списка. Для этого удобно использовать `torch.nn.functional.embedding_bag` с режимом `"mean"`.

### Что нужно реализовать

1. Сохранить переданный слой `embeddings`.
2. В `forward` агрегировать эмбеддинги по спискам с помощью среднего.
3. Вернуть тензор представлений объектов.

### Ожидаемое поведение

Если на вход подан:
- `ids` формы `[num_values, num_hashes]`,
- `lengths` формы `[batch_size]`,

то на выходе должен получиться тензор формы

```python
[batch_size, num_hashes, embedding_dim]
```

То есть для каждого объекта в батче нужно получить по одному усреднённому embedding-представлению для каждого хеша.

In [23]:
# x = torch.randn(2, 3)
# print(x)

# y = torch.randn(2, 3)
# print(y)
# print()
# z = torch.stack([x,y], dim = 0)
# print(z)
# # print(z.ndim)

# z = torch.mean(z, dim = 0)
# print(z)




In [24]:
class MultivalentEncoder(nn.Module):
    def __init__(self, embeddings: nn.Embedding):
        super().__init__()
        self.embeddings = embeddings
      

    def forward(self, ids: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
        offsets = torch.cat([torch.tensor([0], device=ids.device), lengths.cumsum(0)[:-1]])
        
        res = []
        for i in range(ids.shape[1]):  # по каждому хэшу
            col = ids[:, i]            # [num_values] — 1D
            res_temp = torch.nn.functional.embedding_bag(
                col, self.embeddings.weight, offsets, mode='mean'
            )
            res.append(res_temp)       # [batch_size, emb_dim]
        
        return torch.stack(res, dim=1)  # [batch_size, num_hashes, emb_dim]
    


        # embds = self.embeddings(ids)
        # groups = torch.split(embds,lengths.tolist())
        # res = torch.stack([g.mean(dim = 0) for g in groups], dim = 0)
        # return res


tests.test_multivalent_encoder(MultivalentEncoder)

All good! :)


## PiecewiseLinearEncoder

Теперь нужно реализовать `PiecewiseLinearEncoder` — слой для кусочно-линейного кодирования вещественных признаков как в статье [On Embeddings for Numerical Features in Tabular Deep Learning](https://arxiv.org/abs/2203.05556).

Идея этого преобразования такая: каждый вещественный признак разбивается на интервалы по квантилям, а затем значение признака представляется набором кусочно-линейных активаций относительно этих интервалов. В результате один вещественный признак превращается в несколько чисел, которые уже удобнее подавать в табличную нейросетевую модель.

В этом задании вам нужно реализовать:
- `compute_bins`;
- `from_dataset`;
- `__init__`;
- свойство `n_bins`;
- `forward`.

---

### Вход и выход слоя

На вход `forward` подаётся тензор

```python
[batch_size, n_features]
```

где:
- batch_size — число объектов в батче;
- n_features — число вещественных признаков.

На выходе должен получаться тензор

```python
[batch_size, encoded_dim]
```

где encoded_dim — суммарная размерность кусочно-линейного представления всех признаков.

---

Что делает `compute_bins`

Метод `compute_bins(X, n_bins)` должен:
1.	принять тензор X формы `[n_objects, n_features]`;
2.	для каждой колонки отдельно посчитать квантили;
3.	вернуть список тензоров с границами бинов, по одному тензору на каждый признак.

Если бы у всех квантилей были разные значения, то для каждого признака получилось бы ровно n_bins + 1 границ. Но на практике это не всегда так: если в колонке много одинаковых значений, некоторые квантили совпадут. Поэтому после вычисления квантилей нужно удалить повторы через `unique()`.

Именно поэтому реальное число бинов у признака может быть:
- равно n_bins;
- меньше n_bins;
- в крайнем случае равно 1.

---

Что возвращает compute_bins

Метод должен вернуть список длины n_features:

```python
[
    tensor([...]),  # границы бинов для 1-го признака
    tensor([...]),  # границы бинов для 2-го признака
    ...
]
```

Для признака с k бинами тензор границ имеет длину k + 1.

Например:
- если после удаления дублей осталось 5 границ, то это означает 4 бина;
- если осталось 2 границы, то это означает 1 бин.

---

Что делает `from_dataset`

Метод `from_dataset(...)` должен построить готовый объект PiecewiseLinearEncoder по обучающим данным.

Для этого нужно:
1.	взять срез обучающего датафрейма dense_train_df (например, 1_000_000 первых строк);
2.	преобразовать его в тензор;
3.	посчитать биновые границы через compute_bins;
4.	по этим границам построить все внутренние параметры энкодера;
5.	вернуть готовый экземпляр класса.

---

Что такое `weight` и `bias`

Для каждого признака и каждого бина мы заранее строим коэффициенты линейной функции. Если границы очередного бина равны

$$
[a, b],
$$

то для него считаются коэффициенты

$$
w = \frac{1}{b-a}, \qquad c = -\frac{a}{b-a}.
$$

Тогда линейная функция на этом бине имеет вид

$$
w x + c = \frac{x-a}{b-a}.
$$

Именно поэтому всё преобразование потом можно считать векторизованно через

$$
bias + weight \odot x.
$$

Здесь:
- weight хранит наклоны линейных функций;
- bias хранит сдвиги.

---

Почему число бинов может отличаться у разных признаков

После удаления одинаковых квантилей у разных колонок может остаться разное число границ, а значит и разное число бинов.

Например:
- у первого признака может остаться 4 бина;
- у второго — только 2;
- у третьего — только 1.

Чтобы хранить всё в одном тензоре, нужно:
1.	взять максимальное число бинов max_n_bins;
2.	завести weight и bias размера `[n_features, max_n_bins]`;
3.	для признаков с меньшим числом бинов часть позиций оставить пустыми;
4.	затем в forward убрать лишние координаты с помощью mask.

---

Что делает `mask`

`mask` нужен для случая, когда у разных признаков разное число бинов.

Если у всех признаков число бинов одинаковое, то mask можно не использовать и оставить None.

Если число бинов различается, то:
- представление приходится паддить до общего max_n_bins;
- после этого mask указывает, какие координаты нужно оставить, а какие удалить.

Итоговая логика такая:
- сначала строится представление размера [batch_size, n_features, max_n_bins];
- потом оно разворачивается;
- потом лишние паддинговые координаты удаляются через mask.

---

Что делает `single_bin_mask`

single_bin_mask нужен для признаков, у которых после удаления одинаковых квантилей остался ровно один бин.

Это специальный крайний случай. Для таких признаков последняя координата в forward обрабатывается не совсем так же, как для остальных. Поэтому нужен отдельный булевый вектор длины n_features, который отмечает признаки с единственным бином.

Если таких признаков нет, single_bin_mask можно оставить None.

---

Какие крайние случаи нужно учитывать

1. У признака меньше бинов, чем n_bins
Это нормальная ситуация. Она возникает, когда часть квантилей совпала из-за повторяющихся значений. Ничего дополнительно делать не нужно: просто работаем с тем числом бинов, которое реально получилось после `unique()`.

2. У признака ровно один бин
Это тоже допустимая ситуация. Она означает, что после удаления повторов у признака осталось ровно две разные границы.

Такой признак не нужно выбрасывать. Его нужно корректно обработать через single_bin_mask.

3. У признака вообще нет интервала
Если после вычисления квантилей и удаления дублей у признака остаётся только одна уникальная граница, то это означает, что колонка по сути константная и для неё невозможно построить даже один бин.

В этом случае нужно выбросить ошибку:

```python
assert n_bin >= 1, "There is a column with only one unique value"
```

Иными словами:
- 1 бин — допустимо;
- 0 бинов — недопустимо.

4. У всех признаков одинаковое число бинов
В этом случае mask не нужен, и его можно хранить как None.

5. У разных признаков разное число бинов
В этом случае mask обязателен, иначе на выходе останутся лишние координаты от паддинга.

---

Что нужно сделать в `__init__`

В конструкторе нужно:
1.	сохранить список n_bins;
2.	сохранить `weight` и `bias` как buffer;
3.	сохранить mask как buffer, если он не None;
4.	сохранить single_bin_mask как buffer, если он не None.

Здесь именно register_buffer, а не nn.Parameter, потому что это не обучаемые параметры, а заранее посчитанные служебные тензоры.

---

Что должно возвращать свойство `n_bins`

Свойство n_bins должно возвращать список числа бинов для каждого вещественного признака.

Например, если у трёх признаков получилось 4, 2 и 1 бин, то:

```python
encoder.n_bins == [4, 2, 1]
```

---

Что нужно сделать в forward

В forward(x) нужно:
1.	взять вход x формы [batch_size, n_features];
2.	применить векторизованное преобразование через bias и weight;
3.	правильно обрезать значения:
	- первая координата должна быть ограничена сверху;
	- внутренние координаты должны быть ограничены с двух сторон;
	- последняя координата должна обрабатываться отдельно;
4.	учесть специальный случай признаков с одним бином через single_bin_mask;
5.	развернуть последние две размерности;
6.	если есть mask, применить его;
7.	вернуть итоговый тензор признаков.

---

Интуиция про обрезание значений

После вычисления линейных функций получается набор чисел, который нужно превратить в корректное кусочно-линейное представление.

Для этого:
- левая часть должна насыщаться сверху;
- внутренние части должны лежать в диапазоне от 0 до 1;
- правая часть должна насыщаться снизу;
- для признаков с одним бином последняя координата обрабатывается отдельно.

Именно поэтому в forward используются `clamp`, `clamp_min` и `clamp_max`.

---

Подсказка:
- Если не получается решить задачу, то можно обратиться к оригинальной реализации от авторов: https://github.com/yandex-research/rtdl-num-embeddings/tree/a8fc25025c83f2321c63ff127a3bcef83bb1bfb5


In [25]:
# X = torch.randn((3,4))
# print(X)
# n_bins = 3
# intervals = torch.linspace(0,1,n_bins + 1) # влючая и 0 и 1
# q = torch.quantile(X,intervals, dim = 0)
# print(q)
# # print([q[:, i] for i in range(X.shape[1])])
# bins = []
# for i in range(X.shape[1]):
#     col_bins = q[:, i].unique() 
    
#     bins.append(col_bins)
# print(bins)

# # for i, b in enumerate(bins):
# #     print(i,b)


# n_bins_list = [len(b) - 1 for b in bins]
# print(n_bins_list)  

In [26]:
# X = torch.randn((5,6))
# print(X)
# print()
# print(X[1:4])

In [27]:
from torch.nested._internal.ops import slice_tensor
class PiecewiseLinearEncoder(nn.Module):
    @staticmethod
    def compute_bins(X: torch.Tensor, n_bins: int) -> list[torch.Tensor]:
        intervals = torch.linspace(0, 1, n_bins + 1)
        q = torch.quantile(X.float(), intervals, dim=0) 
        bins = []
        for i in range(X.shape[1]):
            col_bins = q[:, i].unique() # ГРАНИЦЫ не бины. Бины = Границы + 1
            assert len(col_bins) >= 2, "There is a column with only one unique value"
            bins.append(col_bins)
        return bins

    @classmethod
    def from_dataset(cls, dense_train_df, n_bins=32, train_df_slice: int = 1_000_000):
        slice_df = dense_train_df[:train_df_slice]
        X = torch.tensor(slice_df.to_numpy(), dtype=torch.float32)

        bins = cls.compute_bins(X, n_bins)
        n_features = len(bins) # кол во признаков
        n_bins_list = [len(b) - 1 for b in bins]   # кол-во промежутков
        max_n_bins = max(n_bins_list) 

        weight = torch.zeros(n_features, max_n_bins)
        bias = torch.zeros(n_features, max_n_bins)

        for i, bin in enumerate(bins):
            for j in range(len(bin)-1):
                a = bin[j]
                b = bin[j + 1]
                weight[i, j] = 1/(b-a)
                bias[i,j] = -a/(b-a)
        
        # маска если не равное колво бинов
        check = all([nb == max_n_bins for nb in n_bins_list])
        if check:
            mask = None
        else:
            mask = torch.zeros(n_features * max_n_bins, dtype=torch.bool)

            for i, bin in enumerate(n_bins_list):
                mask[i*max_n_bins:i*max_n_bins+bin] = True
        

        single_flags = [nb == 1 for nb in n_bins_list]
        single_bin_mask = torch.tensor(single_flags) if any(single_flags) else None
        return cls(weight, bias, mask, n_bins_list, single_bin_mask)

    def __init__(self, weight, bias, mask, n_bins, single_bin_mask):
        super().__init__()
        self._n_bins = n_bins
        self.register_buffer('weight', weight)
        self.register_buffer('bias', bias)

        if mask is not None:
            self.register_buffer('mask', mask)
        else:
            self.mask = None
        if single_bin_mask is not None:
            self.register_buffer('single_bin_mask', single_bin_mask)
        else:
            self.single_bin_mask = None


    @property
    def n_bins(self):
        return self._n_bins

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.unsqueeze(dim=-1) 
        temp_res = x * self.weight + self.bias # broadcasting
        res = torch.cat([temp_res[...,:1].clamp(max=1), temp_res[...,1:].clamp(min=0, max=1)], dim= -1)

        if self.single_bin_mask is not None:
            res = res.clone() 
            res[:, self.single_bin_mask] = res[:, self.single_bin_mask].clamp(0.0, 1.0)

        res = res.flatten(start_dim = 1) 

        if self.mask is not None:
            res = res[:, self.mask] 
        
        return res


tests.test_piecewise_linear_encoder(PiecewiseLinearEncoder)

All good! :)


In [28]:
# # x = torch.tensor([25,45,21])
# x = torch.rand((3,4,2))
# print(x)
# print(x.ndim)
# # x = x.unsqueeze(dim=-1)
# x = x.flatten(start_dim = 1)
# print(x)
# print(x.ndim)

# # w = torch.rand((3,4))
# # b = torch.rand((3,4))

# # temp_res = x * w + b
# # print(temp_res)

# # res = torch.cat([temp_res[:,1].clamp(max=1), col.clamp(min=0,max=1) for col in temp_res[1:].shape[0]])

# # res = torch.cat([temp_res[...,:1].clamp(max=1), temp_res[...,1:].clamp(min=0, max=1)], dim= -1)
# # res = res.flatten(dim = -1)
# # print(res)
# # res= torch.clamp(temp_res,max=1)
# # print(res)
# # print(temp_res[...,:1])
# # print(temp_res[:,1:])



In [29]:
# x = torch.rand(5)
# print(x)
# print(x.unsqueeze(1))

In [30]:
# x = torch.rand((3,4,5))
# print(x)
# print('------')
# print(x[:,2])
# print(x[:2])


## DeepNetwork

Теперь реализуем `DeepNetwork` — обычную полносвязную сеть из линейных слоёв и `ReLU`.

В конструктор передаются:
- `input_dim` — размер входа;
- `hidden_units` — список размеров скрытых слоёв.

Для каждого значения из `hidden_units` нужно добавить `Linear`, а затем `ReLU`. В `forward` нужно просто применить получившуюся сеть к входу `x`.

Если `hidden_units = [h1, h2, ..., hk]`, то для входа формы

```python
[batch_size, input_dim]
```

выход должен иметь форму

```python
[batch_size, hk]
```

In [31]:
class DeepNetwork(nn.Module):
    def __init__(self, input_dim, hidden_units):
        super().__init__()
        self.input_dim = input_dim
        self.hidden_units = hidden_units

        l = []
        prev_dim = input_dim
        for hidden_st in self.hidden_units:
            layer = nn.Linear(prev_dim, hidden_st)
            l.append(layer)
            l.append(nn.ReLU())
            prev_dim = hidden_st


        self.network = nn.Sequential(*l)

        


    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.network(x)

tests.test_deep_network(DeepNetwork)

All good! :)


## ConcatMLP

Теперь нужно реализовать модель `ConcatMLP` — первую полную нейросетевую модель в этом задании.

Идея модели такая:
- sparse-признаки кодируются через `CategoricalEncoder`;
- multivalent-признаки кодируются через `MultivalentEncoder`;
- вещественные признаки кодируются через `PiecewiseLinearEncoder`;
- все полученные представления конкатенируются в один вектор;
- этот вектор подаётся в `DeepNetwork`;
- поверх выхода `DeepNetwork` применяется линейный слой, который выдаёт итоговый score.

### Что передаётся в конструктор

- `embedding_size` — размер эмбеддингов для категориальных и мультивалентных признаков;
- `deep_units` — размеры скрытых слоёв `DeepNetwork`;
- `input_size` — размер входа в `DeepNetwork` после конкатенации всех представлений;
- `dense_train_df` — обучающие вещественные признаки, по которым нужно построить `PiecewiseLinearEncoder`;
- `n_bins` — число бинов для кусочно-линейного кодирования;
- `train_df_slice` — размер среза датафрейма, на котором строится `PiecewiseLinearEncoder`;
- `cardinality` — размер общего embedding-словаря;
- `output_size` — размер выхода модели.

### Что нужно сделать в `__init__`

1. Создать общий слой `nn.Embedding`.
2. Создать:
   - `CategoricalEncoder`,
   - `MultivalentEncoder`,
   - `PiecewiseLinearEncoder`,
   - `DeepNetwork`,
   - финальный `Linear`.
3. Сохранить все эти модули в поля класса.

### Что приходит в `forward`

На вход подаётся словарь `inputs` с той же структурой, что и в `RankerDataset`:

- `inputs["dense_features"]`
- `inputs["sparse_features"]["item_id"]`
- `inputs["sparse_features"]["uid"]`
- `inputs["multivalent_features"]["artist_ids"]["values"]`
- `inputs["multivalent_features"]["artist_ids"]["lengths"]`
- `inputs["multivalent_features"]["album_ids"]["values"]`
- `inputs["multivalent_features"]["album_ids"]["lengths"]`

### Что нужно сделать в `forward`

1. Закодировать `item_id` и `uid` через `CategoricalEncoder`.
2. Закодировать `artist_ids` и `album_ids` через `MultivalentEncoder`.
3. Закодировать dense-признаки через `PiecewiseLinearEncoder`.
4. Развернуть категориальные и мультивалентные представления по последним размерностям.
5. Сконкатенировать все представления по последней оси.
6. Передать результат в `DeepNetwork`.
7. Применить финальный линейный слой.
8. Вернуть итоговый output.

### Ожидаемое поведение

Если на вход подан батч размера `batch_size`, то выход модели должен иметь форму

```python
[batch_size, output_size]

In [32]:
class ConcatMLP(nn.Module):
    def __init__(
        self,
        embedding_size,
        deep_units,
        input_size,
        dense_train_df,
        n_bins,
        train_df_slice,
        cardinality=65536,
        output_size=1,
    ):
        super().__init__() 
        self.embedding = nn.Embedding(cardinality,embedding_size )
        self.categoricalEncoder = CategoricalEncoder(self.embedding)
        self.multivalentEncoder= MultivalentEncoder(self.embedding)
        self.piecewise_encoder = PiecewiseLinearEncoder.from_dataset(
            dense_train_df, n_bins, train_df_slice
        )
        self.deep_network  = DeepNetwork(input_size,deep_units)
        self.output_layer = nn.Linear(deep_units[-1],output_size )


    def forward(self, inputs: dict) -> torch.Tensor:
        catig_item_emb = self.categoricalEncoder(inputs["sparse_features"]["item_id"])
        catig_uid_emb = self.categoricalEncoder(inputs["sparse_features"]["uid"])

        multimodal_artist_emb = self.multivalentEncoder(
            inputs["multivalent_features"]["artist_ids"]["values"],
            inputs["multivalent_features"]["artist_ids"]["lengths"],
        )
        multimodal_album_emb = self.multivalentEncoder(
            inputs["multivalent_features"]["album_ids"]["values"],
            inputs["multivalent_features"]["album_ids"]["lengths"],
        )

        dense_features_emb = self.piecewise_encoder(inputs["dense_features"])

        catig_item_emb = catig_item_emb.flatten(start_dim=1)
        catig_uid_emb = catig_uid_emb.flatten(start_dim=1)
        
        multimodal_artist_emb = multimodal_artist_emb.flatten(start_dim=1)
        multimodal_album_emb = multimodal_album_emb.flatten(start_dim=1)

        total_input = torch.cat([dense_features_emb, catig_item_emb, catig_uid_emb,multimodal_artist_emb,multimodal_album_emb], dim = -1)
        res = self.deep_network(total_input) # total_input - x что в forearde - это инпут 
        res = self.output_layer(res) 

        return res #   batch_size, 1 or 2


In [33]:
# x = torch.rand(2,3,4 )
# print(x)
# print(x.flatten(start_dim=1))

## to_device

Теперь нужно реализовать функцию `to_device`, которая рекурсивно переносит все тензоры в произвольной вложенной структуре данных на заданное устройство.

Такая функция понадобится дальше, чтобы удобно переносить целый батч на CPU или GPU, не разбирая вручную каждое поле.

### Что приходит на вход

- `obj` — произвольный объект;
- `device` — устройство, на которое нужно перенести тензоры.

`obj` может быть:
- `torch.Tensor`;
- `dict`;
- `list`;
- `tuple`;
- любым другим объектом.

### Что нужно сделать

- если `obj` — это `torch.Tensor`, нужно вернуть `obj.to(device)`;
- если `obj` — это `dict`, нужно рекурсивно применить `to_device` ко всем значениям;
- если `obj` — это `list` или `tuple`, нужно рекурсивно применить `to_device` ко всем элементам и сохранить исходный тип контейнера;
- для всех остальных объектов нужно просто вернуть их без изменений.

### Зачем это нужно

Батч в этом задании представляет собой вложенный словарь с dense-, sparse-, multivalent-признаками, таргетами и мета-информацией. Чтобы перед обучением перенести такой батч на нужное устройство, удобно иметь одну универсальную функцию, которая умеет делать это рекурсивно.

In [34]:
def to_device(obj, device: torch.device | str):

    if isinstance(obj, torch.Tensor):
        return obj.to(device)
    elif isinstance(obj, dict):
        return {k: to_device(val, device) for k, val in obj.items()}
    elif isinstance(obj, list):
        return [to_device(i, device) for i in obj]
    elif isinstance(obj, tuple):
        return tuple(to_device(i, device) for i in obj)
    

## Train Loop

Далее вам нужно написать train loop, в котором будет происходить обучение вашей модели. Добавьте в него подсчёт `pairwise accuracy` на тестовом наборе данных после каждой эпохи.

Также предусмотрите поддержку `output_size=1` и `output_size=2`. Это нужно, потому что далее модель будет решать сразу две задачи:
- предсказание лайка;
- предсказание полного прослушивания.

**В случае `output_size=1` обучаемся только на предсказание вероятности полного прослушивания. В случае `output_size=2` на лайк и полное прослушивание.**

Каждая задача должна иметь одинаковый вклад в общую функцию потерь, поэтому никаких дополнительных перевзвешиваний добавлять не нужно. Используйте `BCEWithLogitsLoss`.

В случае `output_size=1` нужно вернуть: 

```python
{
    "pair_accuracy_full_play": ...,
    "pair_accuracy_full_play_like_pairs": ....
}
```

при `output_size=2` нужно вернуть 

```python
{
    "pair_accuracy_full_play": ...,
    "pair_accuracy_full_play_like_pairs": ...
    "pair_accuracy_like": ...,
    "pair_accuracy_like_full_play_pairs": ...
}
```

```
batch = {
    "labels": {
        "is_like":      tensor([0, 1, 0, 1, ...]),        # [4096]
        "is_full_play": tensor([1, 0, 1, 0, ...]),        # [4096]
    },
    "dense_features": tensor([                             # [4096, 15]
        [0.0, 2.0, 100.0, ...],   # 15 счётчиков объекта 0
        [5.0, 0.0,  83.0, ...],   # 15 счётчиков объекта 1
        ...
    ]),
    "sparse_features": {
        "uid":     tensor([[423, 8901],                    # [4096, 2] (2 хэша)
                           [423, 8901], ...]),
        "item_id": tensor([[5120, 33001],                  # [4096, 2]
                           [9821, 12045], ...]),
    },
    "multivalent_features": {
        "artist_ids": {
            "values":  tensor([[102, 5501],                # [num_values, 2]
                               [330, 9921], ...]),
            "lengths": tensor([1, 2, 1, 3, ...]),          # [4096]
        },
        "album_ids": {
            "values":  tensor([[800, 4421],                # [num_values, 2]
                               [110, 6632], ...]),
            "lengths": tensor([1, 1, 2, 1, ...]),          # [4096]
        },
    },
    "meta": {
        "uid":       tensor([100, 100, 200, ...]),         # [4096]
        "item_id":   tensor([732449, 3397170, ...]),       # [4096]
        "timestamp": tensor([40110, 40360, ...]),          # [4096]
    },
}
```

In [ ]:


from torch.nn import BCEWithLogitsLoss
def train_model(
    model,
    train_loader,
    test_loader,
    epochs,
    lr,
    train_log_every,
):
    optimizer = torch.optim.Adam(model.parameters(), lr)
    criterion = BCEWithLogitsLoss() #
    output_size = model.output_layer.out_features
    device = next(model.parameters()).device # 

    for epoch in range(epochs):
        model.train()
        for i, batch in enumerate(train_loader):
            batch = to_device(batch, device)
            output = model(batch) #  batch_size, 1 or 2
    
            if output_size == 1:
                targets = batch["labels"]["is_full_play"].unsqueeze(-1) # batch_size, 1
            else:
                targets = torch.stack([batch["labels"]["is_full_play"], batch["labels"]["is_like"]], dim = -1) #batch_size, 2

            
            loss = criterion(output, targets)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            if i % train_log_every == 0:
                print(f"Epoch {epoch}, batch {i}, loss: {loss.item():.4f}")

    
        model.eval()
        all_scores = []
        all_labels_fp = []
        all_labels_like = []
        all_uids = []
        all_ts = []

        with torch.no_grad():
            for batch in test_loader:
                batch = to_device(batch, device)
                output = model(batch) # логиты batch_size, 1 or 2


                
                all_scores.append(output.cpu()) # num_bacth,  batch_size, 1 or 2
                all_labels_fp.append(batch["labels"]["is_full_play"].cpu()) # batch_size
                all_uids.append(batch["meta"]["uid"].cpu()) # batch_size
                all_ts.append(batch["meta"]["timestamp"].cpu())
                all_labels_like.append(batch["labels"]["is_like"].cpu()) # batch_size

                
                    
        # cклеиваем все батчи в один тензор
        scores = torch.cat(all_scores) # total_obj, 1
        labels_fp = torch.cat(all_labels_fp)  # total_obj
        uids = torch.cat(all_uids) #  # total_obj
        ts = torch.cat(all_ts) # total_obj
        labels_like = torch.cat(all_labels_like)  # [total_obj]
 


        if output_size == 1:
            result = {
                "pair_accuracy_full_play": compute_pairwise_accuracy(
                    uids.numpy(), ts.numpy(), labels_fp.numpy(), scores[:, 0].numpy()
                ),
                "pair_accuracy_full_play_like_pairs": compute_pairwise_accuracy(
                    uids.numpy(),ts.numpy(), labels_like.numpy(), scores[:, 0].numpy()
                )
            }
        else:
            
            result = {
    "pair_accuracy_full_play": compute_pairwise_accuracy(
        uids.numpy(), ts.numpy(), labels_fp.numpy(), scores[:, 0].numpy()
    ),
    "pair_accuracy_full_play_like_pairs": compute_pairwise_accuracy(
        uids.numpy(), ts.numpy(), labels_like.numpy(), scores[:, 0].numpy()
    ),
    "pair_accuracy_like": compute_pairwise_accuracy(
        uids.numpy(), ts.numpy(), labels_like.numpy(), scores[:, 1].numpy()
    ),
    "pair_accuracy_like_full_play_pairs": compute_pairwise_accuracy(
        uids.numpy(), ts.numpy(), labels_fp.numpy(), scores[:, 1].numpy()
    ),
}

        # print(f"Epoch {epoch}: {result}")

    return result 


In [36]:
all_scores = [
    torch.tensor([[0.3], [-0.1], [0.8], [0.5]]),   # батч 0: shape [4, 1]
    torch.tensor([[0.2], [0.9], [-0.3], [0.1]]),    # батч 1: shape [4, 1]
    torch.tensor([[0.7], [-0.5], [0.4], [0.6]]),    # батч 2: shape [4, 1]
]


# a = torch.tensor([[0.3], [-0.1], [0.8], [0.5]])
# print(a.unsqueeze(dim=-1))

x = torch.cat(all_scores, dim = 0)
print(x)

# res = torch.cat(all_scores, dim = 0)
# print(res)


tensor([[ 0.3000],
        [-0.1000],
        [ 0.8000],
        [ 0.5000],
        [ 0.2000],
        [ 0.9000],
        [-0.3000],
        [ 0.1000],
        [ 0.7000],
        [-0.5000],
        [ 0.4000],
        [ 0.6000]])


## Обучение

Вам нужно подобрать параметры обучение так, чтобы пройти тест.

In [ ]:

multihash_transform = MultihashTransform(
    sparse_features_config={
        'item_id': [0, 1],      # 2 хэша с seed 0 и 1
        'uid': [2, 3],          # 2 хэша с seed 2 и 3 (другие seed)
    },
    sparse_features_name="sparse_features",
    multivalent_features_config={
        'artist_ids': [4, 5],   # 2 хэша
        'album_ids': [6, 7],    # 2 хэша
    },
    multivalent_features_name="multivalent_features",
    cardinality=65536,          # совпадает с default в ConcatMLP
)
train_dataset = RankerDataset(
    train_listens,
    [multihash_transform],
    label_columns=list(LABEL_COLUMNS),
    dense_columns=list(DENSE_COLUMNS),
    sparse_columns=list(SPARSE_COLUMNS),
    multivalent_columns=list(MULTIVALENT_COLUMNS),
    batch_size= 4096, #
)

test_dataset = RankerDataset(
    test_listens,
    [multihash_transform],
    label_columns=list(LABEL_COLUMNS),
    dense_columns=list(DENSE_COLUMNS),
    sparse_columns=list(SPARSE_COLUMNS),
    multivalent_columns=list(MULTIVALENT_COLUMNS),
    batch_size=4096, #
)

train_listens_dense_million = train_listens[DENSE_COLUMNS].slice(0, 1_000_000)
del train_listens
del test_listens
gc.collect()

0

In [38]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=1,
    shuffle=True,
    num_workers=0,
    collate_fn=lambda batch: batch[0],
)

test_loader = DataLoader(
    test_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=0,
    collate_fn=lambda batch: batch[0],
)

In [ ]:
device = "mps" if torch.backends.mps.is_available() else "cpu"
# Input size
temp_encoder = PiecewiseLinearEncoder.from_dataset(train_listens_dense_million, n_bins=32)
dense_dim = sum(temp_encoder.n_bins)
embedding_size = 32
num_hashes = 2  # сколько seed на признак (len([0,1]) = 2)
# 4 признака item_id, uid, artist_ids, album_ids
categorical_dim = 4 * num_hashes * embedding_size
input_size = categorical_dim + dense_dim
print(f"dense_dim={dense_dim}, categorical_dim={categorical_dim}, input_size={input_size}")

dense_dim=332, categorical_dim=256, input_size=588


In [ ]:
model_concat_mlp = ConcatMLP(
    embedding_size=32,
    deep_units=[256, 128],
    input_size=input_size,
    dense_train_df=train_listens_dense_million,
    n_bins=32,
    train_df_slice=1_000_000,
    cardinality=65536,
    output_size=1,
)
model_concat_mlp.to(device)

metrics = train_model(
    model=model_concat_mlp,
    train_loader=train_loader,
    test_loader=test_loader,
    epochs=3,
    lr=1e-3,
    train_log_every=100,
)



tests.test_model_concat_mlp_metrics(metrics)


Epoch 0, batch 0, loss: 0.7059
Epoch 0, batch 100, loss: 0.6815
Epoch 0, batch 200, loss: 0.6730
Epoch 0, batch 300, loss: 0.6446
Epoch 0, batch 400, loss: 0.6813
Epoch 0, batch 500, loss: 0.6716
Epoch 0, batch 600, loss: 0.6820
Epoch 0, batch 700, loss: 0.6923
Epoch 0, batch 800, loss: 0.6840
Epoch 0, batch 900, loss: 0.6609
Epoch 0, batch 1000, loss: 0.6731
Epoch 0, batch 1100, loss: 0.6454
Epoch 0, batch 1200, loss: 0.6703
Epoch 0, batch 1300, loss: 0.6636
Epoch 0, batch 1400, loss: 0.6766
Epoch 1, batch 0, loss: 0.6368
Epoch 1, batch 100, loss: 0.6843
Epoch 1, batch 200, loss: 0.6540
Epoch 1, batch 300, loss: 0.6748
Epoch 1, batch 400, loss: 0.5710
Epoch 1, batch 500, loss: 0.6728
Epoch 1, batch 600, loss: 0.6785
Epoch 1, batch 700, loss: 0.6540
Epoch 1, batch 800, loss: 0.6691
Epoch 1, batch 900, loss: 0.6631
Epoch 1, batch 1000, loss: 0.6756
Epoch 1, batch 1100, loss: 0.6774
Epoch 1, batch 1200, loss: 0.6654
Epoch 1, batch 1300, loss: 0.6780
Epoch 1, batch 1400, loss: 0.6820
Epoc

In [69]:
print(metrics)

{'pair_accuracy_full_play': np.float64(0.5873373088008085), 'pair_accuracy_full_play_like_pairs': np.float64(0.48886497493330583)}


# 4. DCN-v2 (3 балла)

В этом задании вам нужно реализовать cross-слой из статьи [DCN V2: Improved Deep & Cross Network and Practical Lessons for Web-scale Learning to Rank Systems](https://arxiv.org/pdf/2008.13535) в варианте **Mixture of Low-Rank Experts**.

Кроме того, дополнительно к `DeepNetwork` вам нужно реализовать ещё две архитектуры:
- `ResDeepNetwork`;
- `DenseDeepNetwork`.

После этого вам нужно будет провести небольшой анализ и сравнить качество разных подходов.

## Mixture Low Rank Cross Layer

Вам нужно реализовать `MixtureLowRankCrossLayer` и `MixtureLowRankCrossNetwork` — low-rank вариант cross-слоя из DCN V2 со смесью экспертов.

Идея этого блока такая: входной вектор признаков несколько раз пропускается через специальные cross-слои, которые моделируют явные взаимодействия между признаками. В отличие от обычного полносвязного слоя, здесь новое представление строится через произведение исходного входа `x0` и преобразования текущего состояния `xl`.

В этой реализации каждый cross-слой состоит из смеси low-rank экспертов:
- каждый эксперт задаёт своё low-rank преобразование;
- затем gate-сеть вычисляет веса экспертов;
- итоговое обновление получается как взвешенная сумма выходов всех экспертов.

### Что нужно реализовать в `MixtureLowRankCrossLayer`

В конструкторе нужно:
1. сохранить `input_dim`, `num_experts` и `rank`;
2. создать обучаемые параметры:
   - `U` формы `[num_experts, input_dim, rank]`,
   - `V` формы `[num_experts, input_dim, rank]`,
   - `bias` формы `[input_dim]`;
3. создать `gate` — линейный слой, который по `xl` предсказывает веса экспертов;
4. инициализировать параметры.

В `forward(x0, xl)` нужно:
1. применить low-rank преобразование к `xl` через `V`, а затем через `U`;
2. прибавить `bias`;
3. умножить результат поэлементно на `x0`;
4. посчитать веса экспертов через `softmax(gate(xl))`;
5. смешать выходы экспертов с этими весами;
6. прибавить residual `xl`.

Итоговый выход должен иметь ту же форму, что и вход:
```python
[batch_size, input_dim]
```

Что нужно реализовать в `MixtureLowRankCrossNetwork`

Это просто стек из нескольких MixtureLowRankCrossLayer.

В конструкторе нужно:
1. создать num_layers cross-слоёв;
2. сохранить их в nn.ModuleList.

В forward(x) нужно:
1. сохранить исходный вход как x0;
2. завести текущее состояние xl = x;
3. последовательно прогнать xl через все cross-слои;
4. вернуть результат последнего слоя.

Важная деталь

Во всей сети x0 остаётся фиксированным и всегда равен исходному входу, а xl меняется от слоя к слою. 

Ожидаемое поведение
- MixtureLowRankCrossLayer принимает x0 и xl формы [batch_size, input_dim] и возвращает тензор той же формы.
- MixtureLowRankCrossNetwork принимает x формы [batch_size, input_dim] и возвращает тензор той же формы.



In [50]:
import torch.nn.functional as F
class MixtureLowRankCrossLayer(nn.Module):
    def __init__(self, input_dim: int, num_experts: int, rank: int):
        super().__init__()

        self.input_dim = input_dim
        self.num_experts = num_experts
        self.rank= rank

        self.U = nn.Parameter(torch.empty(num_experts, input_dim, rank))
        self.V = nn.Parameter(torch.empty(num_experts, input_dim, rank))
        self.bias = nn.Parameter(torch.zeros(input_dim))  # ← zeros, не empty

        nn.init.xavier_uniform_(self.U)
        nn.init.xavier_uniform_(self.V)


        self.gate = nn.Linear(input_dim,num_experts) # работает по последнему dim входа: xl = batch_size x input_dim -> input = input_dim

        # batch_size = b
        #input_dim = d
         #rank = r
        #num_experts = n
       

        
    def forward(self, x0: torch.Tensor, xl: torch.Tensor) -> torch.Tensor:
        # res1 = torch.einsum('bd, ndr -> nbr', xl, self.U)
        # res2 = torch.einsum('nbr, ndr -> nbd', res1, self.V)
        res1 = torch.einsum('bd, ndr -> nbr', xl, self.V)   # V: [batch,dim] → [batch,rank]
        res2 = torch.einsum('nbr, ndr -> nbd', res1, self.U) # U: [batch,rank] → [batch,dim]

        res2 = res2.permute(1, 0, 2)  # [n,b,d] → [b,n,d] 

        

        res3 = res2 + self.bias          # [3,4,5] + [5] → [3,4,5]
        res4 = x0.unsqueeze(1) * res3   # [3,1,5] * [3,4,5] → [3,4,5]

        g = F.softmax(self.gate(xl), dim=-1)  # [3,4]

        g = g.unsqueeze(-1)                    # [3,4,1]
        mixed = (g * res4).sum(dim=1)     # [3,4,1]*[3,4,5] → [3,4,5] → sum → [3,5]
        output = mixed + xl                # [3,5]


        return output


class MixtureLowRankCrossNetwork(nn.Module):
    def __init__(self, input_dim: int, num_layers: int, num_experts: int, rank: int):
        super().__init__()
        self.layers = nn.ModuleList([
            MixtureLowRankCrossLayer(input_dim, num_experts, rank)
            for _ in range(num_layers)
        ])


    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x0 = x
        xl = x
        for l in self.layers:
            xl = l(x0, xl)

        return xl


tests.test_mixture_low_rank_cross_network(MixtureLowRankCrossLayer, MixtureLowRankCrossNetwork)

All good! :)


## ResDeepNetwork

Теперь вам нужно реализовать `ResidualMLPBlock` и `ResDeepNetwork` — residual-вариант глубокой полносвязной сети.

Идея здесь такая: вместо обычной последовательности `Linear -> ReLU -> Linear -> ReLU` каждый блок дополнительно использует skip connection. Это помогает стабилизировать обучение и облегчает прохождение градиентов через глубокую сеть.

### ResidualMLPBlock

`ResidualMLPBlock` — это residual-блок из двух линейных слоёв.

В конструкторе нужно:
- создать первый линейный слой;
- создать второй линейный слой;
- при необходимости добавить проекцию `proj`, если размер входа `in_dim` не совпадает с размером выхода `out_dim`.

В `forward` нужно:
1. пропустить вход через MLP-ветку;
2. прибавить skip connection;
3. если размеры не совпадают, сначала применить `proj` к входу;
4. после сложения применить `ReLU`.

Если `in_dim == out_dim`, то в skip connection можно использовать вход `x` напрямую. Если размеры отличаются, нужно сначала перевести вход в размерность `out_dim` через линейную проекцию.

### ResDeepNetwork

`ResDeepNetwork` — это последовательность residual-блоков.

В конструкторе передаются:
- `input_dim` — размер входа;
- `hidden_units` — список выходных размерностей residual-блоков.

Нужно построить цепочку блоков так, чтобы каждый следующий блок принимал выход предыдущего. Для этого можно завести размеры

```python
[input_dim] + hidden_units
```

и затем создать блоки между соседними размерностями.

В `forward` нужно просто последовательно прогнать вход через все блоки.

Ожидаемое поведение

Если `hidden_units = [h1, h2, ..., hk]`, то для входа формы

```python
[batch_size, input_dim]
```

выход ResDeepNetwork должен иметь форму

```python
[batch_size, hk]
```

   

In [51]:
class ResidualMLPBlock(nn.Module):
    def __init__(self, in_dim: int, out_dim: int):
        super().__init__()
        self.linear1 = nn.Linear(in_dim, out_dim)
        self.linear2 = nn.Linear(out_dim, out_dim)

        if in_dim != out_dim:
            self.flag = 1
            self.proj = nn.Linear(in_dim, out_dim)
        else:
            self.flag = 0 

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if not self.flag:
            res = self.linear2(F.relu(self.linear1(x))) + x
            return F.relu(res)
        
        res = self.linear2(F.relu(self.linear1(x))) + self.proj(x)
        return F.relu(res)



class ResDeepNetwork(nn.Module):
    def __init__(self, input_dim: int, hidden_units: list[int]):
        super().__init__()
        all_dim = [input_dim] + hidden_units
        self.layers = nn.ModuleList([ResidualMLPBlock(indup_dim, output_dim) for indup_dim,output_dim in zip(all_dim[:-1], all_dim[1:])])

        

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        for l in self.layers:
            res = l(x)
            x = res
        return res


tests.test_res_deep_network(ResidualMLPBlock, ResDeepNetwork)

All good! :)


## DenseDeepNetwork

Теперь вам нужно реализовать `DenseDeepNetwork` — полносвязную сеть с dense connectivity.

Идея этой архитектуры такая: каждый следующий слой получает на вход не только выход предыдущего слоя, но и все предыдущие представления, включая исходный вход. То есть на каждом шаге вход в слой строится как конкатенация

```python
x, h_1, h_2, ..., h_{i-1}
```

Это позволяет каждому слою напрямую использовать более ранние признаки и уже посчитанные представления.

При этом важно: конкатенация используется только для формирования входа в следующий слой. Выход самого очередного слоя — это обычный тензор размера, заданного в hidden_units, а не накопленная конкатенация всех прошлых выходов.

Что нужно сделать

В конструкторе передаются:
- input_dim — размер исходного входа;
- hidden_units — список размерностей скрытых слоёв.

Нужно:
1. создать последовательность линейных слоёв;
2. учесть, что размер входа в каждый следующий слой растёт, потому что к нему конкатенируются все предыдущие выходы.

В forward нужно:
1. завести список features, который сначала содержит только исходный вход x;
2. на каждом шаге сконкатенировать все тензоры из features;
3. применить очередной линейный слой и ReLU;
4. добавить новый выход в features;
5. вернуть выход последнего слоя.

Ожидаемое поведение

Если `hidden_units = [h1, h2, ..., hk]`, то для входа формы

```python
[batch_size, input_dim]
```

выход должен иметь форму

```pythonj
[batch_size, hk]
```

In [52]:
class DenseDeepNetwork(nn.Module):
    def __init__(self, input_dim: int, hidden_units: list[int]):
        super().__init__()
        self.layers = nn.ModuleList()
        
        cumulative = input_dim

        for dim in hidden_units:
            self.layers.append(nn.Linear(cumulative, dim))
            cumulative += dim


    def forward(self, x: torch.Tensor) -> torch.Tensor:
        features = [x]
        for layer in self.layers:
            inp = torch.cat(features, dim=1)
            h = F.relu(layer(inp))
            features.append(h)
        return features[-1]


tests.test_dense_deep_network(DenseDeepNetwork)

All good! :)


## DCN V2

Далее вам нужно реализовать итогвую `DCN-v2` модель. Все аналогично `ConcatMLP` модели, но добавляется новый cross-слой, а так же возможность выбрать deep часть. Испольуйте `build_deep_network`.

In [53]:
def build_deep_network(
    input_dim: int,
    hidden_units: list[int],
    deep_type: str = "mlp",
) -> nn.Module:
    """Factory for the deep tower: ``mlp``, ``resnet``, or ``densenet``."""
    if deep_type == "mlp":
        return DeepNetwork(input_dim, hidden_units)
    if deep_type == "resnet":
        return ResDeepNetwork(input_dim, hidden_units)
    if deep_type == "densenet":
        return DenseDeepNetwork(input_dim, hidden_units)
    raise ValueError(f"Unknown deep_type={deep_type!r}, expected 'mlp', 'resnet', or 'densenet'")


In [54]:
class DCNV2(nn.Module):
    def __init__(
        self,
        embedding_size,
        cross_layers,
        deep_units,
        input_size,
        dense_train_df,
        n_bins,
        train_df_slice,
        cardinality=65536,
        num_experts: int = 4,
        low_rank: int = 32,
        deep_network: str = "mlp",
        output_size: int = 2,
    ):
        super().__init__() 
        self.embedding = nn.Embedding(cardinality,embedding_size )
        self.categoricalEncoder = CategoricalEncoder(self.embedding)
        self.multivalentEncoder= MultivalentEncoder(self.embedding)
        self.piecewise_encoder = PiecewiseLinearEncoder.from_dataset(
            dense_train_df, n_bins, train_df_slice
        )
        self.deep_network = build_deep_network(input_size, deep_units, deep_network)
        self.cross_network = MixtureLowRankCrossNetwork(input_size, num_experts, low_rank, cross_layers)

        self.output_layer = nn.Linear(input_size + deep_units[-1], output_size)



    def forward(self, inputs: dict) -> torch.Tensor:
        catig_item_emb = self.categoricalEncoder(inputs["sparse_features"]["item_id"])
        catig_uid_emb = self.categoricalEncoder(inputs["sparse_features"]["uid"])

        multimodal_artist_emb = self.multivalentEncoder(
            inputs["multivalent_features"]["artist_ids"]["values"],
            inputs["multivalent_features"]["artist_ids"]["lengths"],
        )
        multimodal_album_emb = self.multivalentEncoder(
            inputs["multivalent_features"]["album_ids"]["values"],
            inputs["multivalent_features"]["album_ids"]["lengths"],
        )

        dense_features_emb = self.piecewise_encoder(inputs["dense_features"])

        catig_item_emb = catig_item_emb.flatten(start_dim=1)
        catig_uid_emb = catig_uid_emb.flatten(start_dim=1)
        
        multimodal_artist_emb = multimodal_artist_emb.flatten(start_dim=1)
        multimodal_album_emb = multimodal_album_emb.flatten(start_dim=1)

        total_input = torch.cat([dense_features_emb, catig_item_emb, catig_uid_emb,multimodal_artist_emb,multimodal_album_emb], dim = -1)
        
        cross_out = self.cross_network(total_input)    # [batch, input_size]
        deep_out = self.deep_network(total_input)      # [batch, deep_units[-1]]
        combined = torch.cat([cross_out, deep_out], dim=1)  # [batch, input_size + deep_units[-1]]
        res = self.output_layer(combined)
        return res


tests.test_dcnv2(DCNV2)

All good! :)


## Обучение mlp, resnet, densenet с cross-слоями

Ваша задача подобрать параметры для обучения `DCNV2` с `mlp` частью так, чтобы был пройден тест. А так же сравнить этот варинат с `densenet` и `resnet`. 

In [81]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
device


device(type='mps')

In [72]:
model_dcnv2_mlp = DCNV2(
    embedding_size=32,
    deep_units=[256, 128],
    input_size=input_size,
    dense_train_df=train_listens_dense_million,
    n_bins=32,
    train_df_slice=1_000_000,
    cardinality=65536,
    # output_size=1,
    
    cross_layers=2,           # 2-4 
    num_experts=2,            # 2-8
    low_rank=16,              # 16-64
    deep_network="mlp",
).to(device)

metrics = train_model(
    model=model_dcnv2_mlp,
    train_loader=train_loader,
    test_loader=test_loader,
    epochs=3,
    lr=1e-3,
    train_log_every=100,
)

tests.test_model_dcnv2_mlp_metrics(metrics)

Epoch 0, batch 0, loss: 0.6922
Epoch 0, batch 100, loss: 0.5190
Epoch 0, batch 200, loss: 0.4923
Epoch 0, batch 300, loss: 0.3721
Epoch 0, batch 400, loss: 0.3567
Epoch 0, batch 500, loss: 0.3594
Epoch 0, batch 600, loss: 0.3912
Epoch 0, batch 700, loss: 0.5065
Epoch 0, batch 800, loss: 0.4592
Epoch 0, batch 900, loss: 0.3723
Epoch 0, batch 1000, loss: 0.3592
Epoch 0, batch 1100, loss: 0.4226
Epoch 0, batch 1200, loss: 0.3917
Epoch 0, batch 1300, loss: 0.3772
Epoch 0, batch 1400, loss: 0.4544
Epoch 1, batch 0, loss: 0.5118
Epoch 1, batch 100, loss: 0.3634
Epoch 1, batch 200, loss: 0.4546
Epoch 1, batch 300, loss: 0.3712
Epoch 1, batch 400, loss: 0.4291
Epoch 1, batch 500, loss: 0.4100
Epoch 1, batch 600, loss: 0.4710
Epoch 1, batch 700, loss: 0.3942
Epoch 1, batch 800, loss: 0.5074
Epoch 1, batch 900, loss: 0.4918
Epoch 1, batch 1000, loss: 0.5333
Epoch 1, batch 1100, loss: 0.4266
Epoch 1, batch 1200, loss: 0.4947
Epoch 1, batch 1300, loss: 0.4295
Epoch 1, batch 1400, loss: 0.5059
Epoc

In [73]:
metrics

{'pair_accuracy_full_play': np.float64(0.5883870736937191),
 'pair_accuracy_full_play_like_pairs': np.float64(0.48613276970860775),
 'pair_accuracy_like': np.float64(0.5338303712963798),
 'pair_accuracy_like_full_play_pairs': np.float64(0.46047338918663905)}

In [74]:
model_dcnv2_resnet = DCNV2(
    embedding_size=32,
    deep_units=[256, 128],
    input_size=input_size,
    dense_train_df=train_listens_dense_million,
    n_bins=32,
    train_df_slice=1_000_000,
    cardinality=65536,
    #output_size=1,
    
    cross_layers=2,           # 2-4 
    num_experts=2,            # 2-8
    low_rank=16, 
    deep_network="resnet",
).to(device)

metrics = train_model(
    model=model_dcnv2_resnet,
    train_loader=train_loader,
    test_loader=test_loader,
    epochs=3,
    lr=1e-3,
    train_log_every=100,
)


Epoch 0, batch 0, loss: 0.6110
Epoch 0, batch 100, loss: 0.3668
Epoch 0, batch 200, loss: 0.4755
Epoch 0, batch 300, loss: 0.3379
Epoch 0, batch 400, loss: 0.4740
Epoch 0, batch 500, loss: 0.4433
Epoch 0, batch 600, loss: 0.4121
Epoch 0, batch 700, loss: 0.4605
Epoch 0, batch 800, loss: 0.5318
Epoch 0, batch 900, loss: 0.4928
Epoch 0, batch 1000, loss: 0.5562
Epoch 0, batch 1100, loss: 0.3685
Epoch 0, batch 1200, loss: 0.4512
Epoch 0, batch 1300, loss: 0.4167
Epoch 0, batch 1400, loss: 0.4129
Epoch 1, batch 0, loss: 0.4393
Epoch 1, batch 100, loss: 0.4191
Epoch 1, batch 200, loss: 0.3991
Epoch 1, batch 300, loss: 0.4220
Epoch 1, batch 400, loss: 0.3913
Epoch 1, batch 500, loss: 0.3701
Epoch 1, batch 600, loss: 0.4445
Epoch 1, batch 700, loss: 0.4199
Epoch 1, batch 800, loss: 0.4544
Epoch 1, batch 900, loss: 0.3592
Epoch 1, batch 1000, loss: 0.4537
Epoch 1, batch 1100, loss: 0.3966
Epoch 1, batch 1200, loss: 0.3924
Epoch 1, batch 1300, loss: 0.4281
Epoch 1, batch 1400, loss: 0.3678
Epoc

In [75]:
metrics

{'pair_accuracy_full_play': np.float64(0.5885525318451044),
 'pair_accuracy_full_play_like_pairs': np.float64(0.4821504517160053),
 'pair_accuracy_like': np.float64(0.5334952895235395),
 'pair_accuracy_like_full_play_pairs': np.float64(0.46052369740834403)}

In [76]:
model_dcnv2_resnet = DCNV2( # это через densenet
    embedding_size=32,
    deep_units=[256, 128],
    input_size=input_size,
    dense_train_df=train_listens_dense_million,
    n_bins=32,
    train_df_slice=1_000_000,
    cardinality=65536,
   # output_size=1,
    
    cross_layers=2,           # 2-4 
    num_experts=2,            # 2-8
    low_rank=16, 
    deep_network="densenet",
).to(device)

metrics = train_model(
    model=model_dcnv2_resnet,
    train_loader=train_loader,
    test_loader=test_loader,
    epochs=3,
    lr=1e-3,
    train_log_every=100,
)

Epoch 0, batch 0, loss: 0.7674
Epoch 0, batch 100, loss: 0.3905
Epoch 0, batch 200, loss: 0.4592
Epoch 0, batch 300, loss: 0.3969
Epoch 0, batch 400, loss: 0.4370
Epoch 0, batch 500, loss: 0.3752
Epoch 0, batch 600, loss: 0.3974
Epoch 0, batch 700, loss: 0.4209
Epoch 0, batch 800, loss: 0.3879
Epoch 0, batch 900, loss: 0.3670
Epoch 0, batch 1000, loss: 0.4566
Epoch 0, batch 1100, loss: 0.3566
Epoch 0, batch 1200, loss: 0.3810
Epoch 0, batch 1300, loss: 0.5020
Epoch 0, batch 1400, loss: 0.4664
Epoch 1, batch 0, loss: 0.3602
Epoch 1, batch 100, loss: 0.4163
Epoch 1, batch 200, loss: 0.3964
Epoch 1, batch 300, loss: 0.4312
Epoch 1, batch 400, loss: 0.3793
Epoch 1, batch 500, loss: 0.3789
Epoch 1, batch 600, loss: 0.4449
Epoch 1, batch 700, loss: 0.3909
Epoch 1, batch 800, loss: 0.4749
Epoch 1, batch 900, loss: 0.4920
Epoch 1, batch 1000, loss: 0.3554
Epoch 1, batch 1100, loss: 0.3956
Epoch 1, batch 1200, loss: 0.4434
Epoch 1, batch 1300, loss: 0.3910
Epoch 1, batch 1400, loss: 0.3823
Epoc

In [77]:
metrics

{'pair_accuracy_full_play': np.float64(0.5871260142696476),
 'pair_accuracy_full_play_like_pairs': np.float64(0.48904540358021986),
 'pair_accuracy_like': np.float64(0.5312399314371141),
 'pair_accuracy_like_full_play_pairs': np.float64(0.45711727181867573)}

# 5. Многоголовость (2 балла)

Одно из важных преимуществ нейросетей состоит в том, что их можно обучать сразу на несколько таргетов. Например, модель может одновременно решать задачу регрессии (предсказывать число секунд прослушивания) и несколько задач классификации (предсказывать лайк, полное прослушивание и другие типы пользовательского фидбека). На практике таких таргетов может быть довольно много, вплоть до десятков (https://blog.reachsumit.com/posts/2023/04/the-twitter-ml-algo/).

Преимущество такого подхода в том, что модель сразу учится строить общее представление объекта, полезное для разных продуктовых сигналов. После этого уже на этапе анализа или A/B-эксперимента можно подбирать веса разных голов и собирать итоговый скор так, чтобы он лучше соответствовал целям продукта.

В этом задании вы обучите двухголовую нейросетевую модель, которая будет одновременно предсказывать:
- лайк;
- полное прослушивание.

После этого вам нужно будет построить парето-фронт и подобрать такие веса для агрегированного score, чтобы он хорошо работал сразу в двух задачах:
- ранжирование по лайкам;
- ранжирование по полным прослушиваниям.

Если удаётся подобрать хорошие веса, это означает, что итоговое ранжирование одновременно хорошо оптимизирует оба сигнала: пользователи и чаще ставят лайки, и чаще дослушивают треки до конца. Именно такой компромисс обычно и важен для бизнеса.

В конце соберите все результаты в одну таблицу и напишите вывод. 

## Обучение DCNV2 с двумя головами

Вам нужно подобрать параметры обучения так, чтобы пройти тест. В этом задании `output_size=2`, `deep_network='mlp'`.

In [78]:
model_multitask_dcnv2_mlp = DCNV2(
    embedding_size=32,
    deep_units=[256, 128],
    input_size=input_size,
    dense_train_df=train_listens_dense_million,
    n_bins=32,
    train_df_slice=1_000_000,
    cardinality=65536,
    
    cross_layers=2,           # 2-4 
    num_experts=2,            # 2-8
    low_rank=16, 
    deep_network="mlp",
    output_size=2,
).to(device)

metrics = train_model(
    model=model_multitask_dcnv2_mlp,
    train_loader=train_loader,
    test_loader=test_loader,
    epochs=3,
    lr=1e-3,
    train_log_every=100,

)

tests.test_model_multitask_dcnv2_mlp_metrics(metrics)

Epoch 0, batch 0, loss: 0.6760
Epoch 0, batch 100, loss: 0.3904
Epoch 0, batch 200, loss: 0.3851
Epoch 0, batch 300, loss: 0.4969
Epoch 0, batch 400, loss: 0.4266
Epoch 0, batch 500, loss: 0.3843
Epoch 0, batch 600, loss: 0.4298
Epoch 0, batch 700, loss: 0.4114
Epoch 0, batch 800, loss: 0.4610
Epoch 0, batch 900, loss: 0.4808
Epoch 0, batch 1000, loss: 0.3907
Epoch 0, batch 1100, loss: 0.4323
Epoch 0, batch 1200, loss: 0.3592
Epoch 0, batch 1300, loss: 0.4319
Epoch 0, batch 1400, loss: 0.5099
Epoch 1, batch 0, loss: 0.3650
Epoch 1, batch 100, loss: 0.3938
Epoch 1, batch 200, loss: 0.4444
Epoch 1, batch 300, loss: 0.3257
Epoch 1, batch 400, loss: 0.4874
Epoch 1, batch 500, loss: 0.4098
Epoch 1, batch 600, loss: 0.4287
Epoch 1, batch 700, loss: 0.4062
Epoch 1, batch 800, loss: 0.3752
Epoch 1, batch 900, loss: 0.3689
Epoch 1, batch 1000, loss: 0.3738
Epoch 1, batch 1100, loss: 0.3628
Epoch 1, batch 1200, loss: 0.4013
Epoch 1, batch 1300, loss: 0.5130
Epoch 1, batch 1400, loss: 0.4773
Epoc

In [79]:
metrics

{'pair_accuracy_full_play': np.float64(0.5898415402812341),
 'pair_accuracy_full_play_like_pairs': np.float64(0.4836583196937868),
 'pair_accuracy_like': np.float64(0.5320131970667457),
 'pair_accuracy_like_full_play_pairs': np.float64(0.4483748208468327)}

## Результаты

| метод | pair_accuracy_like | pair_accuracy_full_play |
|-------|-------------------:|------------------------:|
| random | 0.503 | 0.500 |
| popular | 0.488 | 0.525 |
| catboost |0.487  |0.591 |
| concatmlp | | 0.587 |
| dcnv2_mlp | 0.534 | 0.588 |
| dcnv2_resnet | 0.533 | 0.589 |
| dcnv2_densenet |0.531 | 0.587 |
| model_multitask_dcnv2_mlp | 0.532 | 0.590 |

Вывод: 

Странно что у многих моделей примерно одинаковые метрики (dcnv2_mlp, dcnv2_resnet, dcnv2_densenet,model_multitask_dcnv2_mlp )

- catboost - сильный бейзлайн 
- Архитектура deep-части не влияет (не знаю, нормально ли это)

- все метрики уперлись full_play уперлись в потолок. 

like метрика гораздо сложнее. Все модели болтаются около рандома (0.5). DCNv2 варианты дают 0.53 — чуть лучше, но catboost (0.487) и popular (0.488) хуже рандома, что странно


## Строим парето-фронт

В этом задании вам нужно построить парето-фронт для двухголовой модели.

В нашем случае парето-фронт — это множество компромиссов между двумя целями:
- качеством ранжирования лайков;
- качеством ранжирования полных прослушиваний.

Идея такая: у модели есть две головы, одна предсказывает вероятность лайка, а другая — вероятность полного прослушивания. Из этих двух предсказаний можно собрать итоговый score как взвешенную сумму

$$
s = \alpha \cdot P(\text{like}) + (1 - \alpha) \cdot P(\text{full\_play}),
$$

где $\alpha \in [0, 1]$.

Важно, что в реальной системе ранжирование всё равно обычно происходит **по одному итоговому score**. Даже если модель предсказывает сразу несколько сигналов, на этапе выдачи объектов нужно отсортировать их по одному числу. Поэтому после обучения многоголовой модели нужно понять, как именно агрегировать выходы разных голов в единый score. Как раз для этого и строится парето-фронт.

При разных значениях $\alpha$ мы получаем разные итоговые ранжирования, а значит и разные значения метрик:
- `pair_accuracy_like`;
- `pair_accuracy_full_play`.

От вас требуется:
1. перебрать разные значения $\alpha$;
2. для каждого значения посчитать итоговый score;
3. вычислить `pair_accuracy_like` и `pair_accuracy_full_play`;
4. построить scatter plot:
   - по оси OX — `pair_accuracy_like`;
   - по оси OY — `pair_accuracy_full_play`.

После этого посмотрите на получившийся график и напишите, какие выводы можно сделать. Используйте хотя бы 100 точек. Возьмите модель из предыдущего пункта. 

In [66]:
import matplotlib.pyplot as plt

#####################
### (づ•̀ᴗ•́)づ──☆*:・ﾟ
#####################

Выводы по графику: 

#####################
### (づ•̀ᴗ•́)づ──☆*:・ﾟ
#####################

## Подберите итоговый score

Теперь вам нужно подобрать такое значение $\alpha$, при котором итоговый score удовлетворяет ограничениям в тесте.

Напомним, что итоговый score строится как

$$
s = \alpha \cdot P(\text{like}) + (1 - \alpha) \cdot P(\text{full\_play}).
$$

На это можно смотреть как на выбор итогового продуктового компромисса: насколько система готова разменивать качество по лайкам на качество по полным прослушиваниям. В реальной задаче именно такой выбор и приходится делать при построении финального ранжирования.

В этом задании вам нужно:
1. подобрать значение $\alpha$;
2. посчитать итоговый score с этим весом;
3. убедиться, что он проходит ограничения из теста.

In [ ]:
alpha =

#####################
### (づ•̀ᴗ•́)づ──☆*:・ﾟ
#####################

metrics = {
    "pair_accuracy_like": compute_pairwise_accuracy(
        #####################
        ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
        #####################
    ),
    "pair_accuracy_full_play": compute_pairwise_accuracy(
        #####################
        ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
        #####################
    ),
}

tests.test_model_multitask_dcnv2_mlp_combined_score_metrics(metrics)